# 01 · Preprocessing

Inspect → harmonize → clean → optionally reserve an unseen attack → split → compress extreme SECOND_BYTES values → fit on train → transform/write each split in batches → release that split.

One dataset per run. Initial loading, splitting, and fitting still need RAM. Saved Parquet files contain ready-to-use transformed features.

## 0. Setup and Imports

In [1]:
PLATFORM = 'kaggle'  # 'colab' or 'kaggle'
assert PLATFORM in ('colab', 'kaggle')

In [2]:
if PLATFORM == 'colab':

    from google.colab import drive
    drive.mount('/content/drive')
    INPATH = '/content/drive/MyDrive/Masters/Datasets/NF-UQ-NIDS(Splitted_Dataset)'
    OUTPATH = '/content/drive/MyDrive/Masters/IDS_repo/My repo'

else :
    INPATH = '/kaggle/input/datasets/omarahmed001/ids-datasets'
    OUTPATH = '/kaggle/working/project'


### Optional GPU startup

Run in a fresh kernel before sklearn/fastai imports. Missing GPU/CuPy/cuML gives a printed CPU mode; no GPU libraries are automatically installed. Compatible cuML uses info logs for actual dispatch/fallback. Other supported preprocessing estimators may also accelerate; numerical/tie differences require validation.


In [3]:
# Run first in a fresh kernel. Compatible cuML/CUDA must already be installed.
%pip install -q "scikit-learn>=1.6" category_encoders imbalanced-learn plotly
import sys
USE_GPU_NEIGHBORS = True
GPU_ACCEL_ENABLED = False
GPU_STATUS = 'CPU (GPU acceleration not requested)'
if USE_GPU_NEIGHBORS:
    try:
        import cupy as cp
        gpu_available = cp.cuda.runtime.getDeviceCount() > 0
    except Exception:
        gpu_available = False
    if not gpu_available:
        GPU_STATUS = 'CPU (NVIDIA GPU/CuPy unavailable)'
    elif 'sklearn' in sys.modules:
        raise RuntimeError('Restart the kernel and run GPU setup before sklearn/fastai imports.')
    else:
        try:
            import cuml.accel
        except ImportError:
            GPU_STATUS = 'CPU (cuML unavailable)'
        else:
            # Installation errors propagate: do not claim CPU fallback after partial activation.
            cuml.accel.install(log_level='info')
            GPU_ACCEL_ENABLED = True
            GPU_STATUS = 'cuML enabled; dispatch logs show actual GPU/CPU execution'
print(GPU_STATUS, flush=True)


Note: you may need to restart the kernel to use updated packages.
CPU (NVIDIA GPU/CuPy unavailable)


In [4]:
from pathlib import Path
from fastai.tabular.all import df_shrink
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, QuantileTransformer
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder
from sklearn.decomposition import PCA
from category_encoders import BinaryEncoder

# Fixed transform: no fitted limits and no row removal.
EXTREME_COLUMNS = ['SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES']

def compress_extreme_values(X):
    """Compress nonnegative SECOND_BYTES measurements before imputation."""
    X = X.copy()
    for col in EXTREME_COLUMNS:
        if col in X.columns:
            values = X[col].astype('float64').replace([np.inf, -np.inf], np.nan)
            if values.lt(0).any():
                raise ValueError(f'{col} contains negative byte measurements.')
            X[col] = np.log1p(values)
    return X

# Display every measured duration as hours, minutes and seconds.
def format_duration(seconds):
    tenths = int(round(float(seconds) * 10))
    hours, remaining = divmod(tenths, 36_000)
    minutes, remaining = divmod(remaining, 600)
    return f'{hours:02d}h {minutes:02d}m {remaining / 10:04.1f}s'


In [5]:
DATA_DIR = Path(INPATH)
FILES = {
    'NF_UNSW_NB15': 'NF-UNSW-NB15-v2.parquet',
    'NF_BOT_IOT': 'NF-BoT-IoT-v2.parquet',
    'NF_CSE_CIC_IDS2018': 'NF-CSE-CIC-IDS2018-v2.parquet',
    'NF_TON_IOT': 'NF-ToN-IoT-v2.parquet',
    'CIC-BoT-IoT': 'CIC-BoT-IoT-V2.parquet',
    'CSE-CIC-IDS2018': 'CSE-CIC-IDS2018.parquet',
    'UNSW-NB15': 'UNSW-NB15.parquet'
}

In [6]:
DATASET = 'NF_TON_IOT'  # Change this to run one of the other datasets.
HOLD_OUT_ATTACK = False  # False: include every attack class in the 70/15/15 split.
HELD_OUT_ATTACK = 'Worms'  # Used only when HOLD_OUT_ATTACK is True.
SEED = 42
MAX_PER_CLASS = 100_000  # Cap before splitting; retain every smaller-class row.


## 1. Load and inspect

In [7]:
df = pd.read_parquet(DATA_DIR / FILES[DATASET])
print(DATASET, df.shape)
display(df.head())
display(df['Attack'].value_counts())

NF_TON_IOT (16940496, 46)


,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,...,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,Label,Attack,Dataset
0,192.168.1.31,42062,192.168.1.79,1041,6,0.0,44,1,40,1,...,0,0,0,0,0,0,0.0,0,Benign,NF-ToN-IoT-v2
1,192.168.1.34,46849,192.168.1.79,9110,6,0.0,44,1,40,1,...,0,0,0,0,0,0,0.0,0,Benign,NF-ToN-IoT-v2
2,192.168.1.30,50360,192.168.1.152,1084,6,0.0,44,1,40,1,...,0,0,0,0,0,0,0.0,0,Benign,NF-ToN-IoT-v2
3,192.168.1.32,56402,192.168.1.169,9012,6,0.0,232,4,132,3,...,65535,0,0,0,0,0,0.0,0,Benign,NF-ToN-IoT-v2
4,192.168.1.31,54001,192.168.1.180,22,6,92.0,84,2,88,2,...,29200,0,0,0,0,0,0.0,1,scanning,NF-ToN-IoT-v2


Attack
Benign        6099469
scanning      3781419
xss           2455020
DDoS          2026234
password      1153323
DoS            712609
injection      684465
Backdoor        16809
mitm             7723
ransomware       3425
Name: count, dtype: int64

In [8]:
df.Label.value_counts()

Label
1    10841027
0     6099469
Name: count, dtype: int64

In [9]:
display(pd.DataFrame({
    'Dtype': df.dtypes,
    'Unique Values': df.nunique()
}))

,Dtype,Unique Values
IPV4_SRC_ADDR,object,23079
L4_SRC_PORT,int64,65536
IPV4_DST_ADDR,object,6868
L4_DST_PORT,int64,65536
PROTOCOL,int64,6
L7_PROTO,float64,213
IN_BYTES,int64,17363
IN_PKTS,int64,2555
OUT_BYTES,int64,15753
OUT_PKTS,int64,1872


### Actual sample versus the full original dataset

Keep up to `MAX_PER_CLASS = 100_000` randomly selected rows per `Attack`; retain every row from classes smaller than the cap. The cells compare this actual sample with the full original data before encoding, cleaning, splitting, or fitted transformations. No second reference sample or 50,000-row diagnostic cap is used.

The class table measures the deliberate change in class proportions. Numerical features are compared within each Attack using KS distance: 0 means matching empirical CDFs; 0.02 means a maximum CDF gap of 2 percentage points. Nominal fields use total variation distance between category frequencies (0 means matching frequencies). Missing and infinite rates are reported separately. Both distances range from 0 to 1; no universal pass/fail cutoff is imposed. Read the complete tables in `numeric_distribution_report` and `categorical_distribution_report`.

These are descriptive comparisons. KS p-values are not reported because the sample is a subset of the original, many features are discrete, and a small p-value would not establish a practically important difference. Marginal agreement does not prove matching correlations, rare subgroups, or tails. Keeping every minority row gives matching raw distributions for that class; it does not add new independent observations.

`CHECK_SAMPLE_DISTRIBUTION = False` skips the feature comparisons while retaining the class table and releasing the original. Full-data comparisons may take time; they process one column at a time and print each class. Rerun **Load and inspect** before repeating sampling.

Sampling remains before the split in this workflow: validation/test therefore reflect the capped class mix. For evaluation at the original class proportions, split the original data first and cap training only. Preprocessing is still fitted after splitting.

In [10]:
import gc

# MAX_PER_CLASS is configured above with DATASET.
CHECK_SAMPLE_DISTRIBUTION = True

# Retain the full original data until the comparison below is finished.
# This assignment does not copy the original DataFrame.
original_df = df
original_counts = original_df['Attack'].value_counts(dropna=False)

sampled_train = pd.concat(
    [
        group.sample(n=min(len(group), MAX_PER_CLASS), random_state=SEED)
        for _, group in original_df.groupby(
            'Attack', observed=True, dropna=False
        )
    ],
    ignore_index=True,
)
df = sampled_train.sample(frac=1, random_state=SEED).reset_index(drop=True)
del sampled_train

sample_counts = df['Attack'].value_counts(dropna=False)
class_comparison = pd.DataFrame({
    'Original rows': original_counts,
    'Sample rows': sample_counts,
}).fillna(0).astype('int64').rename_axis('Attack')
class_comparison['Original (%)'] = (
    100 * class_comparison['Original rows'] / len(original_df)
)
class_comparison['Sample (%)'] = (
    100 * class_comparison['Sample rows'] / len(df)
)
class_comparison['Difference (percentage points)'] = (
    class_comparison['Sample (%)'] - class_comparison['Original (%)']
)
class_comparison['Rows retained (%)'] = (
    100 * class_comparison['Sample rows'] / class_comparison['Original rows']
)
print(f'Original: {len(original_df):,} rows | Actual sample: {len(df):,} rows')
display(class_comparison.round(4))
class_tv_distance = (
    class_comparison['Sample (%)'] - class_comparison['Original (%)']
).abs().sum() / 200
print(f'Class-proportion TV distance: {class_tv_distance:.6f}')

Original: 16,940,496 rows | Actual sample: 727,957 rows


,Original rows,Sample rows,Original (%),Sample (%),Difference (percentage points),Rows retained (%)
Attack,,,,,,
Backdoor,16809,16809,0.0992,2.3091,2.2098,100.0000
Benign,6099469,100000,36.0053,13.7371,-22.2682,1.6395
DDoS,2026234,100000,11.9609,13.7371,1.7762,4.9353
DoS,712609,100000,4.2065,13.7371,9.5305,14.0329
injection,684465,100000,4.0404,13.7371,9.6967,14.6100
mitm,7723,7723,0.0456,1.0609,1.0153,100.0000
password,1153323,100000,6.8081,13.7371,6.9290,8.6706
ransomware,3425,3425,0.0202,0.4705,0.4503,100.0000
scanning,3781419,100000,22.3218,13.7371,-8.5847,2.6445


Class-proportion TV distance: 0.316078


In [11]:
from scipy.stats import ks_2samp

# Nominal fields are compared by category frequencies, not numeric ordering.
# Add another dataset-specific nominal column here if needed.
sampling_nominal_columns = ['PROTOCOL', 'L7_PROTO', 'DNS_QUERY_TYPE', 'FTP_COMMAND_RET_CODE',
                            'ICMP_IPV4_TYPE', 'ICMP_TYPE', 'L4_SRC_PORT', 'L4_DST_PORT', 'TCP_FLAGS',
                            'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'DNS_QUERY_ID']

if CHECK_SAMPLE_DISTRIBUTION:
    categorical_features = [
        c for c in original_df.columns
        if c not in ('Label', 'Attack')
        and (c in sampling_nominal_columns
             or not pd.api.types.is_numeric_dtype(original_df[c]))
    ]
    numeric_features = [
        c for c in original_df.select_dtypes(include='number').columns
        if c not in ('Label', 'Attack') and c not in categorical_features
    ]

    numeric_results = []
    categorical_results = []

    for attack in original_counts.index:
        print(f'Comparing full original vs actual sample: {attack}', flush=True)
        original_mask = (original_df['Attack'].isna() if pd.isna(attack)
                         else original_df['Attack'].eq(attack))
        sample_mask = (df['Attack'].isna() if pd.isna(attack)
                       else df['Attack'].eq(attack))

        # Use every row in both datasets; process one feature at a time.
        for col in numeric_features:
            a = original_df.loc[original_mask, col].to_numpy(
                dtype=np.float64, na_value=np.nan
            )
            b = df.loc[sample_mask, col].to_numpy(
                dtype=np.float64, na_value=np.nan
            )
            finite_a = a[np.isfinite(a)]
            finite_b = b[np.isfinite(b)]

            distance = (
                ks_2samp(finite_a, finite_b, method='asymp').statistic
                if len(finite_a) and len(finite_b) else np.nan
            )
            numeric_results.append({
                'Attack': attack,
                'feature': col,
                'Original rows': len(a),
                'Sample rows': len(b),
                'KS_D': distance,
                'Original missing (%)': 100 * np.isnan(a).mean(),
                'Sample missing (%)': 100 * np.isnan(b).mean(),
                'Original infinite (%)': 100 * np.isinf(a).mean(),
                'Sample infinite (%)': 100 * np.isinf(b).mean(),
            })
            del a, b, finite_a, finite_b

        for col in categorical_features:
            original_frequencies = original_df.loc[
                original_mask, col
            ].value_counts(normalize=True, dropna=False)
            sample_frequencies = df.loc[
                sample_mask, col
            ].value_counts(normalize=True, dropna=False)
            frequencies = pd.concat([
                original_frequencies.rename('original'),
                sample_frequencies.rename('sample'),
            ], axis=1).fillna(0)
            tv_distance = (
                frequencies['original'] - frequencies['sample']
            ).abs().sum() / 2
            categorical_results.append({
                'Attack': attack,
                'feature': col,
                'TV_distance': tv_distance,
            })
            del original_frequencies, sample_frequencies, frequencies

        del original_mask, sample_mask

    numeric_distribution_report = pd.DataFrame(numeric_results, columns=[
        'Attack', 'feature', 'Original rows', 'Sample rows', 'KS_D',
        'Original missing (%)', 'Sample missing (%)',
        'Original infinite (%)', 'Sample infinite (%)',
    ]).sort_values('KS_D', ascending=False)
    categorical_distribution_report = pd.DataFrame(
        categorical_results, columns=['Attack', 'feature', 'TV_distance']
    ).sort_values('TV_distance', ascending=False)

    print('Largest numerical distribution differences (KS distance):')
    display(numeric_distribution_report.head(30))
    if not numeric_distribution_report.empty:
        display(numeric_distribution_report.groupby('Attack', dropna=False)[
            'KS_D'
        ].agg(['median', 'max']))
    print('Categorical distribution differences (total variation distance):')
    display(categorical_distribution_report)
else:
    print('Within-class distribution comparisons disabled.')

# Release the full original before encoding, splitting, and preprocessing.
# The sampled df and the small comparison reports remain available.
del original_df
gc.collect()
print('Sampling comparison finished; original data released from RAM.')

Comparing full original vs actual sample: Benign
Comparing full original vs actual sample: scanning
Comparing full original vs actual sample: xss
Comparing full original vs actual sample: DDoS
Comparing full original vs actual sample: password
Comparing full original vs actual sample: DoS
Comparing full original vs actual sample: injection
Comparing full original vs actual sample: Backdoor
Comparing full original vs actual sample: mitm
Comparing full original vs actual sample: ransomware
Largest numerical distribution differences (KS distance):


,Attack,feature,Original rows,Sample rows,KS_D,Original missing (%),Sample missing (%),Original infinite (%),Sample infinite (%)
136,password,DST_TO_SRC_AVG_THROUGHPUT,1153323,100000,0.004166,0.0,0.0,0.0,0.0
135,password,SRC_TO_DST_AVG_THROUGHPUT,1153323,100000,0.004112,0.0,0.0,0.0,0.0
183,injection,LONGEST_FLOW_PKT,684465,100000,0.003379,0.0,0.0,0.0,0.0
186,injection,MAX_IP_PKT_LEN,684465,100000,0.003379,0.0,0.0,0.0,0.0
122,password,DURATION_OUT,1153323,100000,0.003295,0.0,0.0,0.0,0.0
121,password,DURATION_IN,1153323,100000,0.003252,0.0,0.0,0.0,0.0
137,password,NUM_PKTS_UP_TO_128_BYTES,1153323,100000,0.003236,0.0,0.0,0.0,0.0
176,injection,OUT_BYTES,684465,100000,0.003089,0.0,0.0,0.0,0.0
42,scanning,SRC_TO_DST_SECOND_BYTES,3781419,100000,0.003066,0.0,0.0,0.0,0.0
29,scanning,IN_BYTES,3781419,100000,0.003061,0.0,0.0,0.0,0.0


,median,max
Attack,,
Backdoor,0.000000,0.000000
Benign,0.001064,0.001809
DDoS,0.000943,0.002595
DoS,0.000558,0.001296
injection,0.001616,0.003379
mitm,0.000000,0.000000
password,0.002023,0.004166
ransomware,0.000000,0.000000
scanning,0.000588,0.003066


Categorical distribution differences (total variation distance):


,Attack,feature,TV_distance
16,scanning,L4_SRC_PORT,0.293530
86,DoS,DNS_QUERY_ID,0.270435
18,scanning,L4_DST_PORT,0.263347
41,xss,DNS_QUERY_ID,0.224868
76,DoS,L4_SRC_PORT,0.208861
...,...,...,...
145,ransomware,ICMP_IPV4_TYPE,0.000000
146,ransomware,DNS_QUERY_ID,0.000000
147,ransomware,DNS_QUERY_TYPE,0.000000
148,ransomware,FTP_COMMAND_RET_CODE,0.000000


Sampling comparison finished; original data released from RAM.


## 2. Encoding

The sets below control the `categories` branch. All learned encoders and PCA are fitted on training data only. Keep their raw inputs until transformation.

| Feature | Handling |
|---|---|
| `PROTOCOL`, `L7_PROTO`, `ICMP_IPV4_TYPE`, `DNS_QUERY_TYPE`, `FTP_COMMAND_RET_CODE` | One-hot encoding |
| `IPV4_SRC_ADDR`, `IPV4_DST_ADDR` | Integer encoding, then PCA jointly with `DNS_QUERY_ID` |
| `DNS_QUERY_ID` | Combined with encoded IPs, scaled, then PCA |
| `L4_SRC_PORT`, `L4_DST_PORT` | Binary category encoding |
| `TCP_FLAGS`, `CLIENT_TCP_FLAGS`, `SERVER_TCP_FLAGS` | Eight flag indicators per field |
| `ICMP_TYPE` | Decode the code and one-hot encode it; retain the existing `ICMP_IPV4_TYPE` |

The earlier NF-ToN-IoT check found exact agreement between `ICMP_IPV4_TYPE` and the decoded type on every row. The optional check below can be repeated when changing source datasets. Identifier PCA is an experimental representation: integer IP codes have arbitrary geometry, and PCA does not remove network identity information.

`Encoding(..., scaler=IDENTIFIER_SCALER)` accepts an unfitted scaler instance such as `StandardScaler()`, `MinMaxScaler()` or `QuantileTransformer(output_distribution='normal', random_state=42)`. It scales the three identifier inputs **before PCA**, using its own cloned estimator. One-hot, binary-port and flag outputs are unchanged. `NUMERIC_SCALER` independently controls measurement scaling after imputation; set both variables to the same scaler type if desired. Scalers, learned encoders and PCA are fitted using the full training split. Only IterativeImputer is fitted on a capped training sample, then frozen. QuantileTransformer may internally subsample when estimating quantiles.

`IDENTIFIER_COMPONENTS = 1` compresses the two integer-encoded IPs and `DNS_QUERY_ID` jointly to one output, `categories__pca__pca0`. The fit cell prints the selected scalers and retained variance. Change the component count to 2 or 3 to retain more information. Retained variance refers to the **scaled identifier space**, not attack-prediction quality. Integer IP codes have arbitrary order and DNS query IDs are transaction identifiers, so validate this representation against dropping these identifiers.

MinMax scaling changes ranges but does not reduce outlier influence. Quantile transformation can reduce marginal outlier influence but changes distances/correlations and maps values outside the fitted range to distribution bounds. Scaling before PCA does not bound the final PCA scores. The numeric branch applies log1p to SRC_TO_DST_SECOND_BYTES and DST_TO_SRC_SECOND_BYTES before imputation, then applies the selected measurement scaler. The same fixed compression is used for the imputer fit sample and for every raw-data transformation. It retains rows and ordering, but does not establish whether the original extreme values are valid.

References: [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html), [MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html), [QuantileTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.QuantileTransformer.html).


### Flag decomposition

In [12]:
df['TCP_FLAGS'].value_counts()

TCP_FLAGS
27     208473
0      160699
2      122987
22      80517
19      63785
18      40542
31      13227
16       9552
30       5402
24       5259
26       4890
20       4106
17       3533
25       2727
29        846
28        742
41        183
4         159
21        133
23        126
6          66
219         3
Name: count, dtype: int64

In [13]:
df.CLIENT_TCP_FLAGS.value_counts()

CLIENT_TCP_FLAGS
2      201465
27     193813
0      160701
18      95609
26      20834
30      12773
16      10264
19      10149
24       6180
20       4113
17       3253
25       2875
6        2613
31       2193
28        372
22        293
41        183
4         171
29         42
23         36
21         22
219         3
Name: count, dtype: int64

In [14]:
import pandas as pd

def decompose_tcp_flags(
    df,
    columns=('TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS'),
    inplace=False,
):
    flags = {
        'FIN': 1,
        'SYN': 2,
        'RST': 4,
        'PSH': 8,
        'ACK': 16,
        'URG': 32,
        'ECE': 64,
        'CWR': 128,
    }

    new_columns = {}

    for column in columns:
        values = pd.to_numeric(df[column], errors='raise')

        # Reject missing, fractional, or out-of-range values.
        if (
            values.isna().any()
            or not values.between(0, 255).all()
            or (values % 1 != 0).any()
        ):
            raise ValueError(
                f'{column} must contain non-missing integers from 0 to 255.'
            )

        values = values.to_numpy(dtype='int64')

        for flag, mask in flags.items():
            new_columns[f'{column}_{flag}'] = (
                (values & mask) != 0
            ).astype('uint8')

    result = pd.DataFrame(new_columns, index=df.index)

    if inplace:
        df[result.columns] = result
        return None

    return result

In [15]:
# Flag decomposition is called inside Encoding; do not modify df here.

### ICMP_TYPE decoding into Type and code
* Check reduncy with ICMP_IPV4_TYPE

In [16]:
import pandas as pd

def decode_icmp_type(df, column='ICMP_TYPE', inplace=False):
    values = pd.to_numeric(df[column], errors='raise')

    if (
        values.isna().any()
        or not values.between(0, 65535).all()
        or (values % 1 != 0).any()
    ):
        raise ValueError(
            f'{column} must contain non-missing integers from 0 to 65535.'
        )

    values = values.to_numpy(dtype='int64')

    result = pd.DataFrame({
        'ICMP_TYPE_DECODED': (values // 256).astype('uint8'),
        'ICMP_CODE_DECODED': (values % 256).astype('uint8'),
    }, index=df.index)

    if inplace:
        df[result.columns] = result
        return None

    return result

In [17]:
# Optional diagnostic; the model pipeline decodes ICMP independently.
CHECK_ICMP_REDUNDANCY = False

if CHECK_ICMP_REDUNDANCY:
    icmp_df = decode_icmp_type(df)
    display(icmp_df['ICMP_TYPE_DECODED'].value_counts())

In [18]:
if CHECK_ICMP_REDUNDANCY:
    original_type = pd.to_numeric(df['ICMP_IPV4_TYPE'], errors='raise')
    decoded_type = icmp_df['ICMP_TYPE_DECODED']
    valid = original_type.notna() & decoded_type.notna()
    matches = decoded_type.loc[valid].eq(original_type.loc[valid])

    print('Compared rows:', valid.sum())
    print('Excluded missing rows:', (~valid).sum())
    print('Mismatches:', (~matches).sum())
    if len(matches):
        print(f'Match percentage: {matches.mean():.2%}')
    print('Encoding retains ICMP_IPV4_TYPE and extracts the code from ICMP_TYPE.')

In [19]:
if CHECK_ICMP_REDUNDANCY:
    del icmp_df, original_type, decoded_type, valid, matches
# ICMP_TYPE and ICMP_IPV4_TYPE remain available to Encoding.

In [20]:
# Add only unrelated columns you want removed before splitting.
# Keep identifiers, raw flags, and ICMP inputs for Encoding.
col_to_drop = []

In [21]:
col_to_one_hot = [
    'PROTOCOL',
    'L7_PROTO',
    'ICMP_IPV4_TYPE',
    'DNS_QUERY_TYPE',
    'FTP_COMMAND_RET_CODE',
]

col_to_binary = ['L4_SRC_PORT', 'L4_DST_PORT']
col_to_label = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR']
col_to_pca = col_to_label + ['DNS_QUERY_ID']

col_to_decompose = [
    'TCP_FLAGS',
    'CLIENT_TCP_FLAGS',
    'SERVER_TCP_FLAGS',
]

FLAG_NAMES = ['FIN', 'SYN', 'RST', 'PSH', 'ACK', 'URG', 'ECE', 'CWR']
IDENTIFIER_COMPONENTS = 1  # Three identifier inputs -> one PCA output column.

# Pass scaler INSTANCES. Each pipeline gets its own unfitted copy.
NUMERIC_SCALER = QuantileTransformer(output_distribution='normal', random_state=SEED)  # Measurements, after imputation.
IDENTIFIER_SCALER = QuantileTransformer(output_distribution='normal', random_state=SEED)  # Encoded IPs + DNS_QUERY_ID, before PCA.
# Alternatives for either setting:
# MinMaxScaler()
# QuantileTransformer(output_distribution='normal', random_state=42)
# QuantileTransformer(output_distribution='uniform', random_state=42)
IMPUTER_FIT_ROWS = 500_000  # Sample cap applies only to IterativeImputer.

# Progress and reporting only; no category coverage is enforced.
PREPROCESSING_VERBOSE = True
RUN_CATEGORY_COVERAGE_TESTS = True
PRINT_EVERY_BATCHES = 10  # Also print the first and last batch of each split.

In [22]:
# These are retained as inputs to the identifier PCA branch.
identifiers = col_to_pca.copy()

In [23]:
col_categorical = (
    col_to_one_hot
    + col_to_binary
    + col_to_pca
    + col_to_decompose
    + ['ICMP_TYPE']
)

In [24]:
def prepare_categories(X, flag_cols):
    X = X.copy()
    decompose_tcp_flags(X, columns=flag_cols, inplace=True)
    X['ICMP_CODE_DECODED'] = decode_icmp_type(X)['ICMP_CODE_DECODED']
    return X

def prepared_category_names(transformer, input_features):
    flag_cols = transformer.kw_args['flag_cols']
    return (
        list(input_features)
        + [f'{col}_{flag}' for col in flag_cols for flag in FLAG_NAMES]
        + ['ICMP_CODE_DECODED']
    )

def Encoding(one_hot, binary, label, pca, flags, n_components=1, verbose=True,
             scaler=None):
    """Use the supplied scaler on identifier inputs before PCA only."""
    identifier_scaler = StandardScaler() if scaler is None else clone(scaler)
    flag_columns = [
        f'{col}_{flag}' for col in flags for flag in FLAG_NAMES
    ]
    identifiers = ColumnTransformer([
        ('labels', OrdinalEncoder(
            handle_unknown='use_encoded_value', unknown_value=-1,
        ), label),
        ('numeric', SimpleImputer(strategy='median'),
         [c for c in pca if c not in label]),
    ], verbose=verbose)
    return make_pipeline(
        FunctionTransformer(
            prepare_categories,
            kw_args={'flag_cols': flags},
            feature_names_out=prepared_category_names,
        ),
        ColumnTransformer([
            ('onehot', OneHotEncoder(handle_unknown='ignore'),
             one_hot + ['ICMP_CODE_DECODED']),
            ('binary', BinaryEncoder(
                cols=binary, handle_unknown='value', handle_missing='value',
            ), binary),
            ('flags', 'passthrough', flag_columns),
            ('pca', make_pipeline(
                identifiers, identifier_scaler,
                PCA(n_components=n_components, svd_solver='full'),
                verbose=verbose,
            ), pca),
        ], verbose=verbose),
        verbose=verbose,
    )

## 3. Dataset Shrinking

In [25]:
import warnings
from fastai.tabular.core import df_shrink_dtypes

results = []

for col in df.select_dtypes(include="number").columns:
    # These raw values must stay float64 until the numerical pipeline logs them.
    if col in EXTREME_COLUMNS:
        results.append({
            "column": col,
            "original_dtype": str(df[col].dtype),
            "proposed_dtype": "float64 (preserved)",
            "minimum": df[col].min(),
            "maximum": df[col].max(),
            "warning": "Skipped shrinking; log1p is applied before imputation.",
        })
        continue

    # Inspect one column without copying or modifying the DataFrame.
    skip = [c for c in df.columns if c != col]

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", RuntimeWarning)

        proposed = df_shrink_dtypes(
            df,
            skip=skip,
            obj2cat=False,
            int2uint=False,
        )

    if caught:
        results.append({
            "column": col,
            "original_dtype": str(df[col].dtype),
            "proposed_dtype": str(proposed.get(col, df[col].dtype)),
            "minimum": df[col].min(),
            "maximum": df[col].max(),
            "warning": "; ".join(str(w.message) for w in caught),
        })

display(pd.DataFrame(results))

,column,original_dtype,proposed_dtype,minimum,maximum,warning
0,SRC_TO_DST_SECOND_BYTES,float64,float64 (preserved),0.0,1.120980e+261,Skipped shrinking; log1p is applied before imputation.
1,DST_TO_SRC_SECOND_BYTES,float64,float64 (preserved),0.0,1.872164e+257,Skipped shrinking; log1p is applied before imputation.


In [26]:
big_cols = EXTREME_COLUMNS

for col in big_cols:
    s = df[col]
    print(f"\n{col}")
    print("Outside float32 range:",
          (s.abs() > float(np.finfo(np.float32).max)).sum())
    print(s.quantile([0.5, 0.9, 0.99, 0.999, 1.0]))
    display(df.nlargest(20, col)[
        [col, "IN_BYTES", "OUT_BYTES",
         "FLOW_DURATION_MILLISECONDS", "Attack"]
    ])


SRC_TO_DST_SECOND_BYTES
Outside float32 range: 28
0.500     1.640000e+02
0.900     3.048000e+03
0.990     1.821987e+06
0.999     5.859284e+07
1.000    1.120980e+261
Name: SRC_TO_DST_SECOND_BYTES, dtype: float64


,SRC_TO_DST_SECOND_BYTES,IN_BYTES,OUT_BYTES,FLOW_DURATION_MILLISECONDS,Attack
599849,1.120980e+261,97860,141276,4237573,mitm
416198,6.407206e+210,56160,0,4231797,Benign
470587,3.360176e+165,44520,10764,4258685,mitm
132952,1.120980e+48,420,234,4294795,mitm
492087,1.120980e+48,140,234,4294780,mitm
370101,1.120980e+47,140,234,4294936,mitm
571578,1.120980e+47,280,234,4294654,mitm
669203,1.120980e+47,140,234,4294920,mitm
155772,1.120980e+46,140,234,4294733,mitm
307168,1.120980e+46,140,258,4294280,mitm



DST_TO_SRC_SECOND_BYTES
Outside float32 range: 42
0.500     1.120000e+02
0.900     4.073000e+03
0.990     1.494149e+07
0.999     7.740208e+07
1.000    1.872164e+257
Name: DST_TO_SRC_SECOND_BYTES, dtype: float64


,DST_TO_SRC_SECOND_BYTES,IN_BYTES,OUT_BYTES,FLOW_DURATION_MILLISECONDS,Attack
707182,1.872164e+257,195370,266370,4226618,mitm
599849,2.064181e+223,97860,141276,4237573,mitm
470587,5.382278e+66,44520,10764,4258685,mitm
492087,1.872164e+50,140,234,4294780,mitm
643822,2.613252e+49,156,966,4294503,Benign
132952,1.872164e+49,420,234,4294795,mitm
370101,1.872164e+49,140,234,4294936,mitm
571578,1.872164e+49,280,234,4294654,mitm
669203,1.872164e+49,140,234,4294920,mitm
83347,2.064181e+46,140,258,4294904,mitm


In [27]:
df.columns = df.columns.str.strip()
df['Attack'] = df['Attack'].astype("string[pyarrow]").str.strip()

In [28]:
# Preserve the raw extreme measurements; compress them inside the preprocessor.
df = df_shrink(df, skip=EXTREME_COLUMNS, obj2cat=False, int2uint=False)
df.dtypes

IPV4_SRC_ADDR                           object
L4_SRC_PORT                              int32
IPV4_DST_ADDR                           object
L4_DST_PORT                              int32
PROTOCOL                                  int8
L7_PROTO                               float32
IN_BYTES                                 int32
IN_PKTS                                  int32
OUT_BYTES                                int32
OUT_PKTS                                 int32
TCP_FLAGS                                int16
CLIENT_TCP_FLAGS                         int16
SERVER_TCP_FLAGS                          int8
FLOW_DURATION_MILLISECONDS               int32
DURATION_IN                              int32
DURATION_OUT                             int32
MIN_TTL                                  int16
MAX_TTL                                  int16
LONGEST_FLOW_PKT                         int32
SHORTEST_FLOW_PKT                        int16
MIN_IP_PKT_LEN                           int16
MAX_IP_PKT_LE

## 4. Clean

In [29]:
# Keep all raw encoding inputs, including identifiers.
df = df.drop(columns=['Dataset'], errors='ignore')

In [30]:
print(df.duplicated().sum(), "fully duplicate rows to remove")

0 fully duplicate rows to remove


In [31]:
df.drop_duplicates(inplace=True)
df.reset_index(inplace=True, drop=True)

In [32]:
# Keep Label and Attack for splitting/evaluation; never use them as features.
df = df.loc[:, ~df.columns.str.lower().str.startswith(('unnamed:', 'index'))]
feature_cols = [c for c in df if c not in ('Label', 'Attack')]
categorical_cols = list(col_categorical)
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

# IPs are strings. Numeric protocol/category codes keep their original types.
for col in col_to_label:
    df[col] = df[col].astype('string[pyarrow]').fillna('missing').str.strip()

df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
print(f'{len(numeric_cols)} numeric and {len(categorical_cols)} raw encoding inputs')

29 numeric and 14 raw encoding inputs


In [33]:
print('Categorical: \n', categorical_cols)
print('\n Numeric: \n', numeric_cols)

Categorical: 
 ['PROTOCOL', 'L7_PROTO', 'ICMP_IPV4_TYPE', 'DNS_QUERY_TYPE', 'FTP_COMMAND_RET_CODE', 'L4_SRC_PORT', 'L4_DST_PORT', 'IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'DNS_QUERY_ID', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'ICMP_TYPE']

 Numeric: 
 ['IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'DNS_TTL_ANSWER']


In [34]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print(df.isna().any(axis=1).sum(), "rows with at least one NaN to remove")
# df.dropna(inplace=True)

0 rows with at least one NaN to remove


## 5. Optionally reserve an unseen attack

In [35]:
# Remove incomplete targets and exact repeated rows before any split.
if HOLD_OUT_ATTACK:
    zero_day = df[df['Attack'].eq(HELD_OUT_ATTACK)]
    known = df[df['Attack'].ne(HELD_OUT_ATTACK)]
    print(f'Eligible: {len(known):,} | held-out {HELD_OUT_ATTACK}: {len(zero_day):,}')
else:
    known = df
    zero_day = df.iloc[:0]  # Keep the same schema; no zero-day file is saved.
    print(f'Holdout disabled: all {len(known):,} rows enter the normal split.')


Holdout disabled: all 727,957 rows enter the normal split.


## 6. Split eligible traffic (70% / 15% / 15%)

In [36]:
# Stratify eligible traffic by binary Label; exclude the held-out attack only when enabled.
train, remainder = train_test_split(
    known, test_size=0.30, stratify=known['Attack'], random_state=SEED
)
val, test = train_test_split(
    remainder, test_size=0.50, stratify=remainder['Attack'], random_state=SEED
)
# Drop intermediate frames; retain only the four clean, untransformed splits.
del df, known, remainder
print('train / val / test / zero-day:', *(len(x) for x in (train, val, test, zero_day)))

train / val / test / zero-day: 509569 109194 109194 0


## Outliers

In [37]:
# Only numerical measurements—not categories or flag indicators.
outlier_cols = [
    c for c in numeric_cols
    if c not in col_to_one_hot
    and not c.startswith((
        'TCP_FLAGS_',
        'CLIENT_TCP_FLAGS_',
        'SERVER_TCP_FLAGS_',
    ))
]

rows = []

for col in outlier_cols:
    s = train[col]
    finite = s[np.isfinite(s)]

    q1, median, q3, p99, p999 = finite.quantile(
        [0.25, 0.50, 0.75, 0.99, 0.999]
    )
    iqr = q3 - q1

    # Zero IQR makes this rule uninformative for sparse/zero-heavy features.
    iqr_count = (
        ((finite < q1 - 1.5 * iqr) |
         (finite > q3 + 1.5 * iqr)).sum()
        if iqr > 0 else np.nan
    )

    rows.append({
        'feature': col,
        'missing': s.isna().sum(),
        'infinite': np.isinf(s).sum(),
        'minimum': finite.min(),
        'median': median,
        'p99': p99,
        'p99.9': p999,
        'maximum': finite.max(),
        'IQR': iqr,
        'IQR_flagged': iqr_count,
        'above_p99.9': (finite > p999).sum(),
        'outside_float32': (
            (finite > np.finfo(np.float32).max) |
            (finite < -np.finfo(np.float32).max)
        ).sum(),
        'outside_float64': (
            (finite > np.finfo(np.float64).max) |
            (finite < -np.finfo(np.float64).max)
        ).sum(),
    })

outlier_report = pd.DataFrame(rows).set_index('feature')
display(outlier_report)

,missing,infinite,minimum,median,p99,p99.9,maximum,IQR,IQR_flagged,above_p99.9,outside_float32,outside_float64
feature,,,,,,,,,,,,
IN_BYTES,0,0,4.0,138.0,6.777320e+03,1.309645e+04,1.286171e+07,497.0,58221.0,510,0,0
IN_PKTS,0,0,1.0,2.0,1.000000e+02,2.000000e+02,5.931000e+04,5.0,36830.0,347,0,0
OUT_BYTES,0,0,0.0,100.0,1.281500e+04,2.408146e+04,1.286766e+07,881.0,72171.0,510,0,0
OUT_PKTS,0,0,0.0,1.0,3.000000e+01,1.000000e+02,4.008200e+04,4.0,45741.0,141,0,0
FLOW_DURATION_MILLISECONDS,0,0,0.0,0.0,4.294952e+06,4.294966e+06,4.294966e+06,0.0,NaN,0,0,0
DURATION_IN,0,0,0.0,0.0,7.970000e+02,9.790000e+02,1.190620e+05,0.0,NaN,508,0,0
DURATION_OUT,0,0,0.0,0.0,7.920000e+02,9.570000e+02,6.223900e+04,0.0,NaN,491,0,0
MIN_TTL,0,0,0.0,64.0,2.340000e+02,2.340000e+02,2.550000e+02,64.0,5140.0,93,0,0
MAX_TTL,0,0,0.0,64.0,2.340000e+02,2.340000e+02,2.550000e+02,64.0,5141.0,94,0,0


## 8. Fit preprocessing on training data only
Apply fixed log1p compression to the two SECOND_BYTES measurements before imputation. No rows are removed or percentile limits fitted. Fit IterativeImputer with `skip_complete=True` on at most `IMPUTER_FIT_ROWS` training rows, then wrap it in `FrozenEstimator` so the main fit cannot refit it. Fit the remaining scalers, encoders and identifier PCA using the full training split. Requires scikit-learn 1.6 or newer.

Run from Setup after this change. Old saved preprocessors and prepared datasets must be regenerated.

The main fit still transforms all training rows in memory, including imputation and encoded features. The batch-saving stage below does not batch this fitting step. Category coverage checks remain reports only and never change the split.


In [38]:
from time import perf_counter
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.frozen import FrozenEstimator

print(f'Preparing imputer sample from {len(train):,} training rows...', flush=True)
imputer_sample = train.sample(
    n=min(IMPUTER_FIT_ROWS, len(train)), random_state=SEED,
)[numeric_cols]
imputer_fit_rows = len(imputer_sample)
imputer_sample = compress_extreme_values(imputer_sample)

imputer = IterativeImputer(
    max_iter=5, random_state=SEED, skip_complete=True,
    verbose=int(PREPROCESSING_VERBOSE)
)
print(f'Fitting ONLY IterativeImputer on {imputer_fit_rows:,} rows...', flush=True)
imputer_started = perf_counter()
imputer.fit(imputer_sample)
print(f'Imputer fitted in {format_duration(perf_counter() - imputer_started)}; '
      'freezing it before the remaining fit.', flush=True)
del imputer_sample

preprocessor = ColumnTransformer([
    ('numbers', make_pipeline(
        FunctionTransformer(
            compress_extreme_values, feature_names_out='one-to-one',
        ),
        FrozenEstimator(imputer), clone(NUMERIC_SCALER), verbose=PREPROCESSING_VERBOSE
    ), numeric_cols),
    ('categories', Encoding(
        one_hot=col_to_one_hot,
        binary=col_to_binary,
        label=col_to_label,
        pca=col_to_pca,
        flags=col_to_decompose,
        n_components=IDENTIFIER_COMPONENTS,
        scaler=IDENTIFIER_SCALER,
        verbose=PREPROCESSING_VERBOSE,
    ), categorical_cols),
], verbose=PREPROCESSING_VERBOSE)

print('Measurement scaler:', NUMERIC_SCALER, flush=True)
print('Identifier scaler before PCA:', IDENTIFIER_SCALER, flush=True)
print(f'Identifier PCA: {len(col_to_pca)} inputs -> {IDENTIFIER_COMPONENTS} component(s).', flush=True)

preprocessor_fit_rows = len(train)
print(f'Fitting scalers, encoders and PCA using all {preprocessor_fit_rows:,} training rows: '
      f'{len(numeric_cols)} numeric columns and '
      f'{len(categorical_cols)} encoding inputs. Imputer stays frozen.', flush=True)
fit_started = perf_counter()
preprocessor.fit(train)
print(f'Preprocessing fit finished in {format_duration(perf_counter() - fit_started)}. '
      f'Output features: {len(preprocessor.get_feature_names_out()):,}.', flush=True)
# Diagnostic only: variance is measured after the selected identifier scaling.
fitted_identifier_pipeline = (
    preprocessor.named_transformers_['categories'].steps[-1][1]
    .named_transformers_['pca']
)
fitted_identifier_pca = fitted_identifier_pipeline.steps[-1][1]
identifier_variance_retained = fitted_identifier_pca.explained_variance_ratio_.sum()
print('Identifier PCA variance per component:',
      fitted_identifier_pca.explained_variance_ratio_, flush=True)
print(f'Identifier PCA retains {identifier_variance_retained:.2%} of the scaled identifier variance.',
      flush=True)


Preparing imputer sample from 509,569 training rows...
Fitting ONLY IterativeImputer on 500,000 rows...
[IterativeImputer] Completing matrix with shape (500000, 29)
[IterativeImputer] Change: 0.0, scaled tolerance: 4263376.0 
[IterativeImputer] Early stopping criterion reached.
Imputer fitted in 00h 00m 00.9s; freezing it before the remaining fit.
Measurement scaler: QuantileTransformer(output_distribution='normal', random_state=42)
Identifier scaler before PCA: QuantileTransformer(output_distribution='normal', random_state=42)
Identifier PCA: 3 inputs -> 1 component(s).
Fitting scalers, encoders and PCA using all 509,569 training rows: 29 numeric columns and 14 encoding inputs. Imputer stays frozen.
[Pipeline]  (step 1 of 3) Processing functiontransformer, total=   0.0s
[IterativeImputer] Completing matrix with shape (509569, 29)
[Pipeline] ... (step 2 of 3) Processing frozenestimator, total=   0.5s
[Pipeline]  (step 3 of 3) Processing quantiletransformer, total=   2.4s
[ColumnTransfo

### TEST — category coverage (report only)

Compare all observed unique values across train/validation/test (and optional zero-day) with the full training split and the fitted one-hot/binary/IP vocabularies. This also checks the generated ICMP code. Prints and results do not enforce coverage or change the split. `False` means some values are absent; example values are included. Raw flags, packed ICMP and numeric DNS ID are reported for split/sample coverage but have no learned category vocabulary.

Counts describe the current cleaned/split dataset. Checking evaluation vocabularies is diagnostic only; do not use these results to refit vocabularies on evaluation data. Set `RUN_CATEGORY_COVERAGE_TESTS = False` to skip the full-column scan.

Encoders now fit using every training row. The separate numeric imputer sample does not limit category vocabularies.


In [39]:
# Reporting only: never resplit, resample, refit, or move rows.
if RUN_CATEGORY_COVERAGE_TESTS:
    def category_values(values):
        values = pd.Series(values, copy=False)
        result = set(values.dropna().unique().tolist())
        if values.isna().any():
            result.add(('<missing>',))
        return result

    # Inspect the vocabularies actually learned by the fitted encoders.
    fitted_encoding = preprocessor.named_transformers_['categories'].steps[-1][1]
    learned_values = dict(zip(
        col_to_one_hot + ['ICMP_CODE_DECODED'],
        fitted_encoding.named_transformers_['onehot'].categories_,
    ))
    for mapping in fitted_encoding.named_transformers_['binary'].ordinal_encoder.category_mapping:
        positive_codes = mapping['mapping'] > 0
        learned_values[mapping['col']] = mapping['mapping'].index[positive_codes]
    fitted_ips = fitted_encoding.named_transformers_['pca'].steps[0][1].named_transformers_['labels']
    learned_values.update(dict(zip(col_to_label, fitted_ips.categories_)))

    coverage_splits = {'train': train, 'val': val, 'test': test}
    if HOLD_OUT_ATTACK:
        coverage_splits['zero_day'] = zero_day

    coverage_rows = []
    coverage_columns = categorical_cols + ['ICMP_CODE_DECODED']
    print('Checking full-training and fitted-encoder category coverage one column at a time...', flush=True)

    for number, col in enumerate(coverage_columns, start=1):
        # Decode only the code column; do not build a full transformed dataset.
        if col == 'ICMP_CODE_DECODED':
            split_values = {
                name: category_values(rows['ICMP_TYPE'] % 256)
                for name, rows in coverage_splits.items()
            }
        else:
            split_values = {
                name: category_values(rows[col])
                for name, rows in coverage_splits.items()
            }

        all_values = set().union(*split_values.values())
        train_values = split_values['train']
        missing_train = all_values - train_values
        learned = category_values(learned_values[col]) if col in learned_values else None
        missing_encoder = all_values - learned if learned is not None else None

        coverage_rows.append({
            'column': col,
            'all_unique': len(all_values),
            'train_unique': len(train_values),
            'missing_from_train': len(missing_train),
            'train_covers_all': not missing_train,
            'learned_unique': len(learned) if learned is not None else pd.NA,
            'missing_from_encoder': len(missing_encoder) if learned is not None else pd.NA,
            'encoder_covers_all': not missing_encoder if learned is not None else pd.NA,
            'val_unseen_by_encoder': len(split_values['val'] - learned) if learned is not None else pd.NA,
            'test_unseen_by_encoder': len(split_values['test'] - learned) if learned is not None else pd.NA,
            'zero_day_unseen_by_encoder': len(split_values.get('zero_day', set()) - learned) if learned is not None else pd.NA,
            'missing_train_examples': repr(sorted(missing_train, key=str)[:5]),
            'missing_encoder_examples': repr(sorted(missing_encoder, key=str)[:5]) if learned is not None else '',
        })
        encoder_status = ('not a category vocabulary' if learned is None else
                          f'{len(missing_encoder):,} values absent from fitted encoder')
        print(f'[{number}/{len(coverage_columns)}] {col}: '
              f'train {len(train_values):,}/{len(all_values):,} unique values; '
              f'{encoder_status}.', flush=True)

    category_coverage_report = pd.DataFrame(coverage_rows).set_index('column')
    display(category_coverage_report)
    training_complete = bool(category_coverage_report['train_covers_all'].all())
    encoder_results = category_coverage_report['encoder_covers_all'].dropna()
    encoders_complete = bool(encoder_results.all())
    print('Full training split covers every reported unique value:', training_complete)
    print('Fitted categorical vocabularies cover every observed category:', encoders_complete)
    print('Missing values, if present, are counted as one category.')
    print('Flag combinations and packed ICMP are decoded by fixed rules; '
          'DNS_QUERY_ID enters PCA numerically, so these rows have no learned category vocabulary.')
    if HOLD_OUT_ATTACK:
        print('All_unique includes the held-out attack set; unseen categories there may be intentional.')
    print('Report only: coverage failures do not stop execution or change any split.')
    del coverage_splits



Checking full-training and fitted-encoder category coverage one column at a time...
[1/15] PROTOCOL: train 5/5 unique values; 0 values absent from fitted encoder.
[2/15] L7_PROTO: train 136/141 unique values; 5 values absent from fitted encoder.
[3/15] ICMP_IPV4_TYPE: train 256/256 unique values; 0 values absent from fitted encoder.
[4/15] DNS_QUERY_TYPE: train 11/12 unique values; 1 values absent from fitted encoder.
[5/15] FTP_COMMAND_RET_CODE: train 11/12 unique values; 1 values absent from fitted encoder.
[6/15] L4_SRC_PORT: train 52,252/56,653 unique values; 4,401 values absent from fitted encoder.
[7/15] L4_DST_PORT: train 38,635/46,416 unique values; 7,781 values absent from fitted encoder.
[8/15] IPV4_SRC_ADDR: train 310/420 unique values; 110 values absent from fitted encoder.
[9/15] IPV4_DST_ADDR: train 1,018/1,232 unique values; 214 values absent from fitted encoder.
[10/15] DNS_QUERY_ID: train 51,974/58,265 unique values; not a category vocabulary.
[11/15] TCP_FLAGS: train 

,all_unique,train_unique,missing_from_train,train_covers_all,learned_unique,missing_from_encoder,encoder_covers_all,val_unseen_by_encoder,test_unseen_by_encoder,zero_day_unseen_by_encoder,missing_train_examples,missing_encoder_examples
column,,,,,,,,,,,,
PROTOCOL,5,5,0,True,5,0,True,0,0,0,[],[]
L7_PROTO,141,136,5,False,136,5,False,1,4,0,"[138.0, 213.17799377441406, 231.17799377441406, 7.146999835968018, 79.1780014038086]","[138.0, 213.17799377441406, 231.17799377441406, 7.146999835968018, 79.1780014038086]"
ICMP_IPV4_TYPE,256,256,0,True,256,0,True,0,0,0,[],[]
DNS_QUERY_TYPE,12,11,1,False,11,1,False,1,0,0,[252],[252]
FTP_COMMAND_RET_CODE,12,11,1,False,11,1,False,1,0,0,[220.0],[220.0]
L4_SRC_PORT,56653,52252,4401,False,52252,4401,False,2458,2479,0,"[10, 10001, 10012, 10037, 10039]","[10, 10001, 10012, 10037, 10039]"
L4_DST_PORT,46416,38635,7781,False,38635,7781,False,4238,4195,0,"[10006, 10023, 10032, 10043, 10053]","[10006, 10023, 10032, 10043, 10053]"
IPV4_SRC_ADDR,420,310,110,False,310,110,False,55,55,0,"['102.97.222.182', '104.98.40.113', '109.65.196.0', '110.246.221.63', '112.194.128.122']","['102.97.222.182', '104.98.40.113', '109.65.196.0', '110.246.221.63', '112.194.128.122']"
IPV4_DST_ADDR,1232,1018,214,False,1018,214,False,114,108,0,"['101.119.11.209', '103.126.53.53', '103.6.86.97', '104.16.84.36', '104.18.100.194']","['101.119.11.209', '103.126.53.53', '103.6.86.97', '104.16.84.36', '104.18.100.194']"


Full training split covers every reported unique value: False
Fitted categorical vocabularies cover every observed category: False
Missing values, if present, are counted as one category.
Flag combinations and packed ICMP are decoded by fixed rules; DNS_QUERY_ID enters PCA numerically, so these rows have no learned category vocabulary.
Report only: coverage failures do not stop execution or change any split.


Save each split below in batches, then release its original frame. To repeat saving or run the optional raw-data benchmarks afterward, rerun the loading/splitting cells first.

## 9. Transform, save, and free each split

In [40]:
import pyarrow as pa

feature_names = preprocessor.get_feature_names_out().tolist()

binary_prefixes = (
    'categories__onehot__',
    'categories__binary__',
    'categories__flags__',
)

float64_columns = {
    'numbers__SRC_TO_DST_SECOND_BYTES',
    'numbers__DST_TO_SRC_SECOND_BYTES',
}

original_target_schema = pa.Schema.from_pandas(
    train[['Label', 'Attack']], preserve_index=False
)

schema = pa.schema(
    [
        (
            name,
            pa.uint8() if name.startswith(binary_prefixes)
            else pa.float64() if name in float64_columns
            else pa.float32()
        )
        for name in feature_names
    ]
    + [
        original_target_schema.field('Label'),
        original_target_schema.field('Attack'),
    ]
)

print(f'Compact output schema:\n{schema}')

Compact output schema:
numbers__IN_BYTES: float
numbers__IN_PKTS: float
numbers__OUT_BYTES: float
numbers__OUT_PKTS: float
numbers__FLOW_DURATION_MILLISECONDS: float
numbers__DURATION_IN: float
numbers__DURATION_OUT: float
numbers__MIN_TTL: float
numbers__MAX_TTL: float
numbers__LONGEST_FLOW_PKT: float
numbers__SHORTEST_FLOW_PKT: float
numbers__MIN_IP_PKT_LEN: float
numbers__MAX_IP_PKT_LEN: float
numbers__SRC_TO_DST_SECOND_BYTES: double
numbers__DST_TO_SRC_SECOND_BYTES: double
numbers__RETRANSMITTED_IN_BYTES: float
numbers__RETRANSMITTED_IN_PKTS: float
numbers__RETRANSMITTED_OUT_BYTES: float
numbers__RETRANSMITTED_OUT_PKTS: float
numbers__SRC_TO_DST_AVG_THROUGHPUT: float
numbers__DST_TO_SRC_AVG_THROUGHPUT: float
numbers__NUM_PKTS_UP_TO_128_BYTES: float
numbers__NUM_PKTS_128_TO_256_BYTES: float
numbers__NUM_PKTS_256_TO_512_BYTES: float
numbers__NUM_PKTS_512_TO_1024_BYTES: float
numbers__NUM_PKTS_1024_TO_1514_BYTES: float
numbers__TCP_WIN_MAX_IN: float
numbers__TCP_WIN_MAX_OUT: float
num

# Output folders are retained for checkpoint resume.
# Start a new run folder rather than deleting existing Kaggle outputs here.


In [41]:
from datetime import datetime
from time import perf_counter
import gc
import joblib
import sklearn
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.sparse import issparse

PROJECT = Path(OUTPATH)
fitted_imputer = preprocessor.named_transformers_['numbers'].named_steps['frozenestimator'].estimator
fitted_numeric_scaler = preprocessor.named_transformers_['numbers'].steps[-1][1]
fitted_identifier_pipeline = (
    preprocessor.named_transformers_['categories'].steps[-1][1]
    .named_transformers_['pca']
)
fitted_identifier_scaler = fitted_identifier_pipeline.steps[-2][1]
fitted_identifier_pca = fitted_identifier_pipeline.steps[-1][1]
attack_tag = f"holdout_{HELD_OUT_ATTACK}" if HOLD_OUT_ATTACK else "all_attacks"
RUN_NAME = f"{datetime.now():%Y%m%d_%H%M%S}_{attack_tag}_log1p_{type(fitted_imputer).__name__}"
OUTPUT_DIR = PROJECT / 'outputs' / DATASET / 'preprocessing' / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 10000  # Reduce this if a transformed batch still uses too much RAM.
print('Saving fitted preprocessor to:', OUTPUT_DIR, flush=True)
joblib.dump(preprocessor, OUTPUT_DIR / 'preprocessor.joblib', compress=0)
print('Fitted preprocessor saved.', flush=True)


# Small exact baseline from the raw TRAIN split, before its frames are released.
# Counts include missing as a distinct entry. Keep only 20 example values per field.
import json
raw_category_rows = []
for column in categorical_cols:
    unique_values = train[column].drop_duplicates()
    raw_category_rows.append({
        'Raw feature': column,
        'Raw unique entries': len(unique_values),
        'Raw values (first 20)': unique_values.head(20).astype(str).tolist(),
        'Raw missing rows': int(train[column].isna().sum()),
    })
    del unique_values
raw_category_lookup = {r['Raw feature']: r for r in raw_category_rows}
encoded_category_rows = []
for column in preprocessor.get_feature_names_out():
    if not column.startswith('categories__'):
        continue
    if column.startswith('categories__pca__'):
        inputs, encoding = list(col_to_pca), 'Joint identifier PCA'
    elif column.startswith('categories__onehot__ICMP_CODE_DECODED_'):
        inputs, encoding = ['ICMP_TYPE'], 'Decoded ICMP code + one-hot'
    else:
        branch, encoding, candidates = next(
            (branch, encoding, candidates)
            for branch, encoding, candidates in [
                ('onehot', 'One-hot', col_to_one_hot),
                ('binary', 'Binary category bits', col_to_binary),
                ('flags', 'TCP flag bits', col_to_decompose),
            ] if column.startswith(f'categories__{branch}__')
        )
        inputs = [c for c in candidates if column.startswith(f'categories__{branch}__{c}_')]
    encoded_category_rows.append({
        'Raw feature(s)': ' + '.join(inputs),
        'Raw unique entries': {c: raw_category_lookup[c]['Raw unique entries'] for c in inputs},
        'Raw values (preview)': {c: raw_category_lookup[c]['Raw values (first 20)'] for c in inputs},
        'Encoding': encoding,
        'Encoded column': column,
    })
(OUTPUT_DIR / 'raw_categorical_summary.json').write_text(json.dumps({
    'training_rows': len(train),
    'sampling_max_per_class': MAX_PER_CLASS,
    'seed': SEED,
    'raw_features': raw_category_rows,
    'encoded_features': encoded_category_rows,
}, indent=2), encoding='utf-8')
del raw_category_rows, raw_category_lookup, encoded_category_rows

# Move frames into the queue; remove the other references so they can be freed.
splits = {'train': train, 'val': val, 'test': test}
if HOLD_OUT_ATTACK:
    splits['zero_day'] = zero_day
del train, val, test, zero_day
counts = []
for name in list(splits):
    rows = splits.pop(name)
    split_started = perf_counter()
    total_batches = (len(rows) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f'Starting {name}: {len(rows):,} rows in {total_batches:,} batches.', flush=True)
    with pq.ParquetWriter(OUTPUT_DIR / f'{name}.parquet', schema, compression='zstd') as writer:
        for start in range(0, len(rows), BATCH_SIZE):
            batch = rows.iloc[start:start + BATCH_SIZE]
            X = preprocessor.transform(batch[feature_cols])
            # # Reduce batch memory before densifying sparse output.
            # # float32 rounding is intentional for compact model inputs.
            # X = X.astype(np.float32, copy=False)
            # Densify this small batch only; never the complete split.
            if issparse(X):
                X = X.toarray()
            prepared = pd.DataFrame(X, columns=feature_names)
            prepared['Label'] = batch['Label'].to_numpy()
            prepared['Attack'] = batch['Attack'].to_numpy()
            table = pa.Table.from_pandas(prepared, schema=schema, preserve_index=False)
            writer.write_table(table)
            batch_number = start // BATCH_SIZE + 1
            if (batch_number == 1 or batch_number == total_batches
                    or batch_number % max(1, PRINT_EVERY_BATCHES) == 0):
                completed = min(start + BATCH_SIZE, len(rows))
                print(f'{name}: batch {batch_number:,}/{total_batches:,}; '
                      f'{completed:,}/{len(rows):,} rows written; '
                      f'{format_duration(perf_counter() - split_started)} elapsed.', flush=True)
            del batch, X, prepared, table
    counts.append(f'- {name}: {len(rows):,} rows')
    print(f'Saved {name}: {len(rows):,} rows in {format_duration(perf_counter() - split_started)}; releasing raw split.', flush=True)
    del rows
    gc.collect()  # Frees Python references; the allocator may retain some reserved RAM.

holdout_note = f"Reserve `{HELD_OUT_ATTACK}` as zero-day data" if HOLD_OUT_ATTACK else "Use all attack classes; no separate zero-day set"
saved_files = '`train.parquet`, `val.parquet`, `test.parquet`' + (', `zero_day.parquet`' if HOLD_OUT_ATTACK else '')
readme = f"""# {DATASET} preprocessing — {RUN_NAME}

Source: `{DATA_DIR / FILES[DATASET]}`
Holdout enabled: `{HOLD_OUT_ATTACK}`. Held-out attack: `{HELD_OUT_ATTACK if HOLD_OUT_ATTACK else "None (all attacks included)"}`. Seed: `{SEED}`. scikit-learn: `{sklearn.__version__}`.

## Steps followed
1. Inspect data and attack counts; strip column/attack-name whitespace.
2. Remove Dataset, configured drop columns, and generated index columns; retain IPs and DNS_QUERY_ID for identifier PCA.
3. Keep categorical numeric codes separate from measurements; normalize IP strings and replace numeric infinities with NaN.
4. Remove exact duplicate rows.
5. {holdout_note}; split eligible traffic 70/15/15, stratified by Attack. With holdout disabled, attack classes are eligible for every split; rare classes are not guaranteed to occur in each split.
6. Apply fixed log1p compression to `{EXTREME_COLUMNS}` before imputation, without clipping or deleting rows. Fit only `{fitted_imputer!r}` on {imputer_fit_rows:,} sampled training rows (cap {IMPUTER_FIT_ROWS:,}), then freeze it. Fit the remaining preprocessing using all {preprocessor_fit_rows:,} training rows: `{fitted_numeric_scaler!r}` for measurements; Encoding combines one-hot codes, binary ports, decomposed TCP flags, decoded ICMP code, and integer IP encoding plus identifier scaling with `{fitted_identifier_scaler!r}` then PCA ({fitted_identifier_pca.n_components_} components).
7. Transform each split in batches of at most {BATCH_SIZE:,} rows and write directly to Parquet with Zstandard compression.
8. Release each batch and completed split before continuing.

## Identifier PCA diagnostics
Input columns: `{col_to_pca}`. Retained variance after identifier scaling: `{fitted_identifier_pca.explained_variance_ratio_.sum():.2%}`.
Variance is a compression diagnostic, not a measure of attack-prediction quality. IP ordinal mappings are arbitrary and DNS query IDs are transaction identifiers.
Scaling before PCA does not bound PCA outputs. The two SECOND_BYTES measurements are log1p-compressed before imputation. Measurement scaling follows imputation.

## Progress and category coverage
PREPROCESSING_VERBOSE controls fit-step timings. PRINT_EVERY_BATCHES controls batch-write progress.
RUN_CATEGORY_COVERAGE_TESTS enables a report comparing full-training/evaluation vocabularies and actual fitted encoders. Only the numeric imputer uses a training sample; encoders use the full training split.
Checks do not enforce coverage, refit on evaluation data, resample, or modify any split.

## Runtime, checkpoints and GPU
All measured durations shown by this notebook use hours/minutes/seconds. Numeric seconds are retained internally in checkpoints. The resampling workflow uses four small shared helpers; optional prepared dtypes/outlier inspections are under Testing.
This NF_TON_IOT version caps sampled rows at 100,000 per Attack before the split. Main ADASYN selects classes below 0.5 of the largest TRAIN class and targets 0.5 of that count, with a 3% tolerance around the requested target. Native class names and binary Label remain separate from model features.
Per-policy resampling/ directories hold full uncompressed ADASYN checkpoints, validated using SHA256 of prepared training content, feature order, seed, cap, policy parameters and package versions. GPU backend status is recorded; restored times describe original computation, not reload time. Optional cuML activation runs before sklearn imports, logs actual dispatch/fallback, and may also accelerate other supported estimators. CPU/GPU neighbours can differ at ties or due to numerical arithmetic.
Outputs and checkpoints are saved in OUTPUT_DIR. Save Kaggle outputs before discarding a session.
The optional RUN_THRESHOLD_TESTS block uses main prepared train/validation inputs for an unresampled baseline and paired selection/target policies 0.25, 0.5 and 0.75. Each uses the same 32-unit MLP/Adam/30-epoch settings. Test data is never used by those comparisons. Results/models/confusion matrices are saved below threshold_tests/. Completed-result reuse and original compute timings are labelled explicitly.

## Saved files
`raw_categorical_summary.json` records exact raw-training categorical domains and their encoded-column mappings for Testing. The imbalance cells also save `adasyn_generated_preview.parquet` with a small genuine generated-row preview and Tomek retention flags; its metadata columns are not model inputs.
{saved_files} contain TRANSFORMED numeric feature columns plus `Label` and `Attack`.
Features are already log-compressed where configured, imputed, encoded, and scaled: do not transform them again.
`preprocessor.joblib` is for new RAW data only. Feature order is `preprocessor.get_feature_names_out()`.
Storage types: one-hot, binary-port and flag outputs use uint8; identifier PCA and ordinary scaled measurements use float32. The two configured SECOND_BYTES outputs retain float64 storage.
Label and Attack retain their original Arrow types. The same schema is used for every batch and split.
Float32 reduces precision compared with float64.
These files differ from older exports of cleaned, untransformed features.
To load preprocessor.joblib in a new session, first run the imports, EXTREME_COLUMNS and compress_extreme_values definition, decoder definitions, column sets and Encoding helper definitions; do not fit again.
IP code mappings and PCA are learned from training only. Identifier PCA does not guarantee generalization across networks.
The two SECOND_BYTES features use fixed log1p compression before imputation. This prevents their original magnitudes from reaching the regression models, but does not prove that the original measurements are valid. No outlier rows are removed. Complete columns are skipped by IterativeImputer during fitting.

{chr(10).join(counts)}

## Read prepared batches
```python
import pyarrow.parquet as pq
for record_batch in pq.ParquetFile('train.parquet').iter_batches(batch_size=10000):
    batch = record_batch.to_pandas()
    X = batch.drop(columns=['Label', 'Attack'])
    y = batch['Label']
    # Consume this batch directly; do not apply preprocessor.transform(X).
```

Initial loading/splitting and imputer fitting still require RAM. Fitting the remaining preprocessing transforms the full training split in memory; the later batch-saving loop does not batch fitting. FrozenEstimator requires scikit-learn 1.6 or newer. Saving releases the original split frames; rerun loading/splitting before using raw-data benchmarks or repeating the save step.
Row splits and exact deduplication do not guarantee session isolation or isolation of identical predictors with conflicting labels.
"""
(OUTPUT_DIR / 'README.md').write_text(readme, encoding='utf-8')
print('Prepared datasets saved to:', OUTPUT_DIR)

Saving fitted preprocessor to: /kaggle/working/project/outputs/NF_TON_IOT/preprocessing/20261116_112333_all_attacks_log1p_IterativeImputer
Fitted preprocessor saved.
Starting train: 509,569 rows in 51 batches.
[IterativeImputer] Completing matrix with shape (10000, 29)
train: batch 1/51; 10,000/509,569 rows written; 00h 00m 00.5s elapsed.
[IterativeImputer] Completing matrix with shape (10000, 29)
[IterativeImputer] Completing matrix with shape (10000, 29)
[IterativeImputer] Completing matrix with shape (10000, 29)
[IterativeImputer] Completing matrix with shape (10000, 29)
[IterativeImputer] Completing matrix with shape (10000, 29)
[IterativeImputer] Completing matrix with shape (10000, 29)
[IterativeImputer] Completing matrix with shape (10000, 29)
[IterativeImputer] Completing matrix with shape (10000, 29)
[IterativeImputer] Completing matrix with shape (10000, 29)
train: batch 10/51; 100,000/509,569 rows written; 00h 00m 04.3s elapsed.
[IterativeImputer] Completing matrix with shap

## 7. Class Imbalance

Main order: split → log1p on the two SECOND_BYTES fields → impute/encode/scale → transform/save → ADASYN on train → Tomek Links → model training.

Load the processed `train.parquet`, resample its `Attack` classes with ADASYN, then clean the original majority Attack class with Tomek Links. Exclude both targets from features. Restore binary `Label` from the original Attack-to-Label mapping when saving. Validation and test are not resampled here. Synthetic one-hot coordinates can be fractional and remain floating point.
Run the cells below in order: settings → shared loading/preview tools → checkpoint tools → restore checkpoint → load training if needed → ADASYN → save checkpoint → Tomek → tables → plot → save balanced train. Only four reused helpers remain here. All measured durations are displayed as `00h 00m 00.0s`; numeric seconds remain internal to checkpoint calculations. Optional dtype/outlier inspections are under Testing.


In [42]:
HANDEL_IMBALANCE = True
SELECT_BELOW_RATIO = 0.50
TARGET_RATIO = 0.50
ADASYN_BALANCE_TOLERANCE = 0.03  # Skip negligible top-ups around the requested target.
REUSE_ADASYN_CHECKPOINT = True
if not 0 < SELECT_BELOW_RATIO <= TARGET_RATIO <= 1:
    raise ValueError('Require 0 < selection ratio <= target ratio <= 1.')
# Fresh-session resume: set an EXISTING output run here; do not rerun main export.
RESUME_OUTPUT_DIR = None  # e.g. '/kaggle/working/project/outputs/NF_TON_IOT/preprocessing/<run>'
if RESUME_OUTPUT_DIR is not None:
    OUTPUT_DIR = Path(RESUME_OUTPUT_DIR)


### Shared loading and sample-preview tools

These two functions are reused by the main run and the three threshold comparisons. The batched loader retains the memory guard and checks the Attack-to-Label mapping; the preview contains genuine appended ADASYN rows.


In [43]:
from pathlib import Path
from time import perf_counter
import gc
import json
import math
import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import plotly.express as px
import psutil
import sklearn
import imblearn
from imblearn.over_sampling import ADASYN
from imblearn.under_sampling import TomekLinks

# Load a single dense matrix in batches; keep fractional synthetic coordinates.
def read_prepared_matrix(path, feature_names=None):
    parquet = pq.ParquetFile(path)
    schema = parquet.schema_arrow
    names = feature_names or [c for c in schema.names if c not in ('Label', 'Attack')]
    dtype = np.result_type(np.float32, *[
        np.dtype(schema.field(c).type.to_pandas_dtype()) for c in names
    ])
    rows = parquet.metadata.num_rows
    required = rows * (len(names) * dtype.itemsize + np.dtype(object).itemsize)
    available = psutil.virtual_memory().available
    print(f'Loading {rows:,} rows × {len(names):,} features; dtype={dtype}; '
          f'working arrays ≈ {required / 2**30:.2f} GiB; '
          f'available RAM={available / 2**30:.2f} GiB', flush=True)
    if required > available * 0.5:
        raise MemoryError('Working arrays exceed half the available RAM. Reduce training rows.')
    X = np.empty((rows, len(names)), dtype=dtype)
    y = np.empty(rows, dtype=object)
    target_parts = []
    start = 0
    for record in parquet.iter_batches(batch_size=20_000, columns=names + ['Attack', 'Label'], use_threads=False):
        frame = record.to_pandas()
        stop = start + len(frame)
        for j, column in enumerate(names):
            X[start:stop, j] = frame[column].to_numpy(copy=False)
        y[start:stop] = frame['Attack'].to_numpy(copy=False)
        target_parts.append(frame[['Attack', 'Label']].drop_duplicates())
        start = stop
    pairs = pd.concat(target_parts, ignore_index=True).drop_duplicates()
    if pairs['Attack'].duplicated().any():
        raise ValueError('An Attack class has multiple Label values.')
    metadata = {
        'original_columns': schema.names,
        'target_dtypes': pairs.dtypes,
        'attack_to_label': pairs.set_index('Attack')['Label'],
    }
    return X, y, names, metadata

def generated_row_preview(X, y, input_rows, targets, names, label_map):
    positions = []
    for attack in targets:
        positions.extend((input_rows + np.flatnonzero(y[input_rows:] == attack)[:5]).tolist())
    columns = [c for c in names if c.startswith('categories__')]
    indices = [names.index(c) for c in columns]
    preview = pd.DataFrame(X[np.ix_(positions, indices)], columns=columns)
    preview.insert(0, 'ADASYN row', positions)
    preview['Attack'] = y[positions]
    preview['Label'] = preview['Attack'].map(label_map)
    preview['Kept after Tomek'] = pd.Series([None] * len(preview), dtype='boolean')
    return preview


### Checkpoint tools

Two reused functions calculate the prepared-file fingerprint and save a complete checkpoint atomically. They preserve compatibility with existing ADASYN checkpoints.


In [44]:
import hashlib

_file_signatures = {}

def prepared_signature(path):
    """Hash once per changed file; content identity survives copying the prepared output run."""
    path = Path(path)
    stat = path.stat()
    key = (str(path.resolve()), stat.st_size, stat.st_mtime_ns)
    if key not in _file_signatures:
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
                digest.update(block)
        _file_signatures[key] = digest.hexdigest()
    return _file_signatures[key]

def atomic_checkpoint(path, state, settings):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.tmp')
    joblib.dump({'settings': settings, 'state': state}, temporary, compress=0)
    temporary.replace(path)
    print('Checkpoint saved:', path, flush=True)


### 1. Prepare settings and restore a matching ADASYN checkpoint

For fresh-session resume, run Setup (including format_duration), dataset/sampling constants, imbalance settings and the two utility cells. Set RESUME_OUTPUT_DIR to an existing prepared run, then run this cell. A matching checkpoint skips training-array loading and ADASYN. All source/settings compatibility checks remain; mismatches require REUSE_ADASYN_CHECKPOINT=False to recompute.

This cell also prepares the identity needed by the optional threshold/model tests, including when main resampling is disabled. Keep the complete prepared output run when moving between sessions.


In [45]:
# For a fresh session, set OUTPUT_DIR to the existing prepared run before this cell.
# Run Setup, imbalance settings, the two utility cells, and this restore cell first.
# This identity is also used by the optional prepared-data threshold tests.
current_policy_settings = {
    'format': 1, 'train_sha256': prepared_signature(OUTPUT_DIR / 'train.parquet'),
    'features': [c for c in pq.read_schema(OUTPUT_DIR / 'train.parquet').names
                 if c not in ('Label', 'Attack')],
    'dataset': DATASET, 'seed': SEED, 'max_per_class': MAX_PER_CLASS,
    'selection': SELECT_BELOW_RATIO, 'target': TARGET_RATIO,
    'tolerance': ADASYN_BALANCE_TOLERANCE, 'max_neighbors': 5,
    'sklearn': sklearn.__version__, 'imblearn': imblearn.__version__,
}
ADASYN_CACHE_LOADED = False
if HANDEL_IMBALANCE:
    RESAMPLING_DIR = OUTPUT_DIR / 'resampling' / f'select_{SELECT_BELOW_RATIO:g}_target_{TARGET_RATIO:g}'
    RESAMPLING_DIR.mkdir(parents=True, exist_ok=True)
    ADASYN_CHECKPOINT = RESAMPLING_DIR / 'adasyn_checkpoint.joblib'
    if REUSE_ADASYN_CHECKPOINT and ADASYN_CHECKPOINT.exists():
        checkpoint = joblib.load(ADASYN_CHECKPOINT, mmap_mode='r')
        if checkpoint['settings'] != current_policy_settings:
            raise ValueError('Checkpoint settings/data differ. Set REUSE_ADASYN_CHECKPOINT=False to recompute.')
        restored_state = checkpoint['state']
        globals().update(restored_state)
        del restored_state, checkpoint
        ADASYN_CACHE_LOADED = True
        print(f'Restored ADASYN checkpoint: X={X_adasyn.shape}, y={y_adasyn.shape}', flush=True)
        print('Original ADASYN compute time:', format_duration(adasyn_seconds), '| backend:', adasyn_backend)
        generated_preview.to_parquet(OUTPUT_DIR / 'adasyn_generated_preview.parquet', index=False)


In [46]:
if HANDEL_IMBALANCE and not ADASYN_CACHE_LOADED:
    X_train, y_train, resampling_features, train_metadata = read_prepared_matrix(
        OUTPUT_DIR / 'train.parquet',
    )
    original_columns = train_metadata['original_columns']
    target_dtypes = train_metadata['target_dtypes']
    attack_to_label = train_metadata['attack_to_label']
    before_counts = pd.Series(y_train).value_counts().sort_index()
    majority_attack = before_counts.idxmax()
    display(before_counts.rename_axis('Attack').rename('rows').to_frame())
    del train_metadata


Loading 509,569 rows × 510 features; dtype=float64; working arrays ≈ 1.94 GiB; available RAM=29.28 GiB


,rows
Attack,
Backdoor,11766
Benign,70000
DDoS,70000
DoS,70000
injection,70000
mitm,5406
password,70000
ransomware,2397
scanning,70000


### 2. Select minority classes and run ADASYN

Selection and requested counts are calculated directly in the next code cell. With the default 0.5/0.5 policy, select classes below half the largest TRAIN class and request half its count; skip additions within 3% of that requested count. Unselected classes stay in the reference data. ADASYN failures remain visible without substituting another sampler.

Start/end output, post-ADASYN shape, added rows and matrix size remain visible. GPU dispatch is reported by startup/cuML logs. Cached times describe the original computation.


### ADASYN — timing, shape and generated-row preview


In [47]:
if HANDEL_IMBALANCE and not ADASYN_CACHE_LOADED:
    input_train_rows = len(y_train)
    print(f'Before ADASYN: X={X_train.shape}, y={y_train.shape}', flush=True)
    adasyn_seconds = 0.0
    adasyn_skipped = False
    requested_count = math.ceil(TARGET_RATIO * int(before_counts.max()))
    sampling_targets = {
        attack: requested_count for attack, count in before_counts.items()
        if count < SELECT_BELOW_RATIO * before_counts.max()
        and count < (1 - ADASYN_BALANCE_TOLERANCE) * requested_count
    }
    print('Selected class targets:', sampling_targets, flush=True)

    if sampling_targets:
        smallest_target_class = min(
            int(before_counts[attack]) for attack in sampling_targets
        )
        if smallest_target_class < 2:
            raise ValueError(
                'ADASYN needs at least two examples in each targeted Attack class.'
            )

        adasyn = ADASYN(
            sampling_strategy=sampling_targets,
            n_neighbors=min(5, smallest_target_class - 1),
            random_state=SEED,
        )
        adasyn_started = perf_counter()
        print('ADASYN: starting...', flush=True)
        X_adasyn, y_adasyn = adasyn.fit_resample(X_train, y_train)
        adasyn_seconds = perf_counter() - adasyn_started
    else:
        adasyn_skipped = True
        print('No classes require the selected partial-balancing policy; skipping ADASYN.', flush=True)
        X_adasyn, y_adasyn = X_train, y_train

    adasyn_rows = len(y_adasyn)
    print(f'After ADASYN: X={X_adasyn.shape}, y={y_adasyn.shape}', flush=True)
    print(f'ADASYN time: {format_duration(adasyn_seconds)} | '
          f'Rows added: {adasyn_rows - input_train_rows:,}', flush=True)
    print(f'Post-ADASYN feature matrix: {X_adasyn.nbytes / 2**30:.2f} GiB | '
          f'dtype={X_adasyn.dtype}', flush=True)

    adasyn_counts = pd.Series(y_adasyn).value_counts().sort_index()
    print('After ADASYN (Attack):')
    display(pd.DataFrame({
        'Before': before_counts,
        'After ADASYN': adasyn_counts,
    }).fillna(0).astype(int).rename_axis('Attack'))

    adasyn_backend = GPU_STATUS if sampling_targets else 'skipped'
    generated_preview = generated_row_preview(
        X_adasyn, y_adasyn, input_train_rows, sampling_targets,
        resampling_features, attack_to_label,
    )
    generated_preview.to_parquet(OUTPUT_DIR / 'adasyn_generated_preview.parquet', index=False)

    del X_train, y_train
    gc.collect()

Before ADASYN: X=(509569, 510), y=(509569,)
Selected class targets: {'Backdoor': 35000, 'mitm': 35000, 'ransomware': 35000}
ADASYN: starting...
After ADASYN: X=(595206, 510), y=(595206,)
ADASYN time: 00h 03m 17.5s | Rows added: 85,637
Post-ADASYN feature matrix: 2.26 GiB | dtype=float64
After ADASYN (Attack):


,Before,After ADASYN
Attack,,
Backdoor,11766,34998
Benign,70000,70000
DDoS,70000,70000
DoS,70000,70000
injection,70000,70000
mitm,5406,35193
password,70000,70000
ransomware,2397,35015
scanning,70000,70000


### 3. Save the full ADASYN checkpoint

Atomic local save. This checkpoint contains features, targets, counts and preview metadata, retaining the numerical matrix dtype. An interrupted temporary write is not accepted as a completed checkpoint.


In [48]:
if HANDEL_IMBALANCE and not ADASYN_CACHE_LOADED:
    cache_variables = [
        'X_adasyn', 'y_adasyn', 'resampling_features', 'original_columns',
        'target_dtypes', 'attack_to_label', 'before_counts', 'adasyn_counts',
        'majority_attack', 'input_train_rows', 'adasyn_rows', 'adasyn_seconds',
        'adasyn_skipped', 'generated_preview', 'sampling_targets', 'adasyn_backend',
    ]
    cache_state = {name: globals()[name] for name in cache_variables}
    atomic_checkpoint(ADASYN_CHECKPOINT, cache_state, current_policy_settings)
    del cache_state
if HANDEL_IMBALANCE:
    settings_path = RESAMPLING_DIR / 'policy_settings.json'
    settings_path.write_text(json.dumps(current_policy_settings, indent=2))


Checkpoint saved: /kaggle/working/project/outputs/NF_TON_IOT/preprocessing/20261116_112333_all_attacks_log1p_IterativeImputer/resampling/select_0.5_target_0.5/adasyn_checkpoint.joblib


### 4. Tomek Links — timing and output shape


In [49]:
if HANDEL_IMBALANCE:
    print(f'Before Tomek Links: X={X_adasyn.shape}, y={y_adasyn.shape}', flush=True)
    # Preserve the original policy: clean only the original majority class.
    tomek = TomekLinks(
        sampling_strategy=[majority_attack],
        n_jobs=-1,
    )
    tomek_started = perf_counter()
    print('Tomek Links: starting...', flush=True)
    X_train_balanced, y_train_balanced = tomek.fit_resample(X_adasyn, y_adasyn)
    tomek_seconds = perf_counter() - tomek_started
    tomek_removed_rows = len(y_adasyn) - len(y_train_balanced)
    print(f'After Tomek Links: X={X_train_balanced.shape}, '
          f'y={y_train_balanced.shape}', flush=True)
    print(f'Tomek Links time: {format_duration(tomek_seconds)} | '
          f'Rows removed: {tomek_removed_rows:,}', flush=True)
    after_counts = pd.Series(y_train_balanced).value_counts()

    # Record which preview rows survived, using Tomek's actual selected indices.
    generated_preview['Kept after Tomek'] = np.isin(
        generated_preview['ADASYN row'].to_numpy(), tomek.sample_indices_,
    )
    generated_preview.to_parquet(
        OUTPUT_DIR / 'adasyn_generated_preview.parquet', index=False,
    )
    del generated_preview

    # Free the expanded input before plotting or exporting the balanced output.
    del X_adasyn, y_adasyn
    gc.collect()


Before Tomek Links: X=(595206, 510), y=(595206,)
Tomek Links: starting...
After Tomek Links: X=(594442, 510), y=(594442,)
Tomek Links time: 01h 44m 18.2s | Rows removed: 764


### 5. Resampling results and timings


In [50]:
if HANDEL_IMBALANCE:
    distribution = pd.DataFrame({
        'Before': before_counts,
        'After ADASYN': adasyn_counts,
        'After Tomek Links': after_counts,
    }).fillna(0).astype(int).rename_axis('Attack')

    distribution['Before (%)'] = (
        100 * distribution['Before'] / distribution['Before'].sum()
    ).round(2)
    distribution['After (%)'] = (
        100 * distribution['After Tomek Links']
        / distribution['After Tomek Links'].sum()
    ).round(2)
    display(distribution)

    resampling_summary = pd.DataFrame([
        {'Stage': 'ADASYN', 'Status': 'cached' if ADASYN_CACHE_LOADED else ('skipped' if adasyn_skipped else 'finished'),
         'Input rows': input_train_rows, 'Output rows': adasyn_rows,
         'Features': len(resampling_features), 'Time': format_duration(adasyn_seconds)},
        {'Stage': 'Tomek Links', 'Status': 'finished',
         'Input rows': adasyn_rows, 'Output rows': adasyn_rows - tomek_removed_rows,
         'Features': len(resampling_features), 'Time': format_duration(tomek_seconds)},
    ]).set_index('Stage')
    display(resampling_summary)
    distribution_path = RESAMPLING_DIR / 'class_distribution.csv'
    distribution.to_csv(distribution_path)
    timing_path = RESAMPLING_DIR / 'timings.csv'
    resampling_summary.to_csv(timing_path)


,Before,After ADASYN,After Tomek Links,Before (%),After (%)
Attack,,,,,
Backdoor,11766,34998,34998,2.31,5.89
Benign,70000,70000,69236,13.74,11.65
DDoS,70000,70000,70000,13.74,11.78
DoS,70000,70000,70000,13.74,11.78
injection,70000,70000,70000,13.74,11.78
mitm,5406,35193,35193,1.06,5.92
password,70000,70000,70000,13.74,11.78
ransomware,2397,35015,35015,0.47,5.89
scanning,70000,70000,70000,13.74,11.78


,Status,Input rows,Output rows,Features,Time
Stage,,,,,
ADASYN,finished,509569,595206,510,00h 03m 17.5s
Tomek Links,finished,595206,594442,510,01h 44m 18.2s


### 6. Plot class distribution

Run this cell again to redraw the chart without rerunning Tomek Links.


In [51]:
if HANDEL_IMBALANCE:
    # Plot aggregated counts instead of passing millions of rows to Plotly.
    plot_counts = distribution[
        ['Before', 'After ADASYN', 'After Tomek Links']
    ].reset_index().melt(
        id_vars='Attack', var_name='Stage', value_name='Rows'
    )
    plot_counts['Attack'] = plot_counts['Attack'].astype(str)
    fig = px.histogram(
        plot_counts,
        x='Attack',
        y='Rows',
        color='Stage',
        histfunc='sum',
        barmode='group',
        title='Attack class distribution before and after resampling',
        labels={'Rows': 'Number of samples'},
        category_orders={'Attack': distribution.index.astype(str).tolist()},
    )
    fig.update_layout(yaxis_title='Number of samples', xaxis_tickangle=-45)
    plot_path = RESAMPLING_DIR / 'class_distribution.html'
    fig.write_html(plot_path, include_plotlyjs='cdn')
    fig.show()



### 7. Save the balanced training data


In [52]:
if HANDEL_IMBALANCE:
    balanced_train = pd.DataFrame(
        X_train_balanced,
        columns=resampling_features,
    )
    balanced_train['Attack'] = pd.Series(y_train_balanced).astype(
        target_dtypes['Attack']
    )
    balanced_train['Label'] = balanced_train['Attack'].map(
        attack_to_label
    ).astype(target_dtypes['Label'])
    balanced_train = balanced_train[original_columns]

    balanced_path = OUTPUT_DIR / 'train_adasyn_tomek.parquet'
    balanced_train.to_parquet(
        balanced_path,
        index=False,
        compression='zstd',
    )
    print('Saved:', balanced_path)

    del balanced_train, X_train_balanced, y_train_balanced
    gc.collect()

Saved: /kaggle/working/project/outputs/NF_TON_IOT/preprocessing/20261116_112333_all_attacks_log1p_IterativeImputer/train_adasyn_tomek.parquet


Train using `train_adasyn_tomek.parquet`, with the existing `val.parquet` and `test.parquet`. The saved training file contains both `Attack` and `Label`; use `Attack` as the multiclass target. Exclude both targets from the model inputs. These features are already preprocessed—do not transform them again.

The table and Plotly histograms above show counts before resampling, after ADASYN, and after Tomek Links. Percentages compare the original and final training distributions.

# Testing

Run Setup and Imports (including `compress_extreme_values`) and the configuration cells at the top of the notebook (platform, paths, dataset and seed), then start here. The loading cell applies log1p once to the two SECOND_BYTES fields before any benchmark imputation or scaling. You can also run this section after the main save cell has released its raw splits. Enable the desired tests below, then run the remaining Testing cells in order.

The first loading cell reloads the **same raw Parquet selected by `DATA_DIR / FILES[DATASET]`**, repeats the relevant feature preparation, and recreates the original 70/15/15 split. Only its training partition enters these experiments. It does not require the main fitted preprocessor or the main in-memory splits.

The categorical diagnostic below uses the main saved outputs and its own flag. It needs `OUTPUT_DIR` for a completed main export/resampling run and is independent of `RUN_TESTING`/`RUN_COMPARISONS`. The legacy raw-data loading cell below remains for the existing imputer/model benchmarks only.

The new prepared-data threshold/model comparison is independent of the legacy raw-data benchmarks. Run startup/imports, dataset/sampling constants and imbalance settings, the two utility cells and the checkpoint-restore cell, then point OUTPUT_DIR to a completed prepared run. RUN_THRESHOLD_TESTS uses train.parquet and val.parquet only; it does not require the released raw splits or existing main resampling arrays.


In [53]:
# Enable either or both groups, then run the loading cell immediately below.
RUN_TESTING = False      # Existing imputer benchmarks.
RUN_COMPARISONS = False # Encoding -> scaler -> imbalance comparisons, including the paper's method.
RUN_CATEGORY_DIAGNOSTICS = True  # Main raw -> encoded -> resampled categorical audit.

RUN_THRESHOLD_TESTS = True  # Baseline + paired 0.25/0.5/0.75 policies; can take time.

RUN_PREPARED_INSPECTIONS = False  # Optional saved dtypes/outlier inspection; needs outlier_cols from main.


### Optional saved dtypes and outlier inspection

Enable RUN_PREPARED_INSPECTIONS only when needed. The outlier inspection requires outlier_cols from the main preprocessing run. These inspections are independent of ADASYN and are not needed to resume a checkpoint.


In [54]:
if RUN_PREPARED_INSPECTIONS:
    import pandas as pd
    import pyarrow.parquet as pq

    stored_schema = pq.read_schema(OUTPUT_DIR / 'train.parquet')

    dtype_map = {
        field.name: field.type.to_pandas_dtype()
        for field in stored_schema
    }

    display(
        pd.Series(dtype_map, name='dtype')
        .astype(str)
        .rename_axis('column')
        .to_frame()
    )

In [55]:
if RUN_PREPARED_INSPECTIONS:
    import numpy as np
    import pandas as pd
    import pyarrow.parquet as pq

    train_path = OUTPUT_DIR / 'train.parquet'
    saved_columns = set(pq.read_schema(train_path).names)

    rows = []

    for original_col in outlier_cols:
        original_col = original_col.removeprefix('numbers__')

        # Exclude binary flag indicators.
        if original_col.startswith((
            'TCP_FLAGS_',
            'CLIENT_TCP_FLAGS_',
            'SERVER_TCP_FLAGS_',
        )):
            continue

        col = f'numbers__{original_col}'

        if col not in saved_columns:
            print(f'Skipped: {col} is not in the prepared training file.')
            continue

        # Load only this feature, rather than all 486 columns.
        s = pd.read_parquet(train_path, columns=[col])[col]

        # Convert only this column's finite values for the calculations.
        finite = s[np.isfinite(s)].astype(np.float64, copy=False)

        q1, median, q3, p99, p999 = finite.quantile(
            [0.25, 0.50, 0.75, 0.99, 0.999]
        )
        iqr = q3 - q1

        iqr_count = (
            ((finite < q1 - 1.5 * iqr) |
             (finite > q3 + 1.5 * iqr)).sum()
            if iqr > 0 else np.nan
        )

        rows.append({
            'feature': col,
            'dtype': str(s.dtype),
            'missing': s.isna().sum(),
            'infinite': np.isinf(s).sum(),
            'minimum': finite.min(),
            'median': median,
            'p99': p99,
            'p99.9': p999,
            'maximum': finite.max(),
            'IQR': iqr,
            'IQR_flagged': iqr_count,
            'above_p99.9': (finite > p999).sum(),
            'outside_float32': (
                (finite > np.finfo(np.float32).max) |
                (finite < -np.finfo(np.float32).max)
            ).sum(),
            'outside_float64': (
                (finite > np.finfo(np.float64).max) |
                (finite < -np.finfo(np.float64).max)
            ).sum(),
        })

        del s, finite

    outlier_report = pd.DataFrame(rows).set_index('feature')
    display(outlier_report)

### Categorical unique values and generated samples

Enable `RUN_CATEGORY_DIAGNOSTICS` and use `OUTPUT_DIR` from the main run. This test reads saved diagnostics and training files; it does not rerun ADASYN/Tomek or use the legacy benchmark encoding below.

- **Before processing:** exact raw categorical unique counts from the same training split, saved before its raw frames are released. Up to 20 raw entries are retained as examples. Counts include missing as a distinct entry.
- **After processing:** exact unique counts per encoded output in `train.parquet`.
- **After resampling:** exact unique counts, new-entry counts and example values per encoded output in `train_adasyn_tomek.parquet` (ADASYN followed by Tomek). `MAX_UNIQUE_EXAMPLES` controls the displayed encoded-value preview; the counts use every row. One projected column is read at a time to limit memory, so large files may take time.
- **Generated samples:** up to five genuine appended ADASYN rows per targeted Attack, saved before cleanup, with their encoded coordinates and whether Tomek retained them. These are before-Tomek generated examples, not randomly selected final rows. The small preview may miss fractional values present elsewhere; the full-data table checks final indicator domains.

One-hot columns, binary-port bits and flag indicators normally contain 0/1 before resampling. ADASYN can interpolate them to fractional coordinates. The diagnostic displays those values without rounding or changing any data. Identifier PCA combines the IP/DNS fields jointly into continuous outputs; its non-integer values are expected and are not interpreted as category violations or mapped back to a single raw category. This compares value domains, not category frequencies or proof of semantic validity.

For an older output run missing the diagnostic files, rerun the main export while raw `train` and the fitted preprocessor are available, then the ADASYN/Tomek cells. Raw splits and resampling matrices are normally released, so reloading/splitting/fitting may be needed to recreate them. No existing artifacts are silently relabelled as synthetic.


In [56]:
if RUN_CATEGORY_DIAGNOSTICS:
    import json
    import numpy as np
    import pandas as pd
    import pyarrow as pa
    import pyarrow.compute as pc
    import pyarrow.parquet as pq

    # OUTPUT_DIR must point to the same run for all three diagnostic files.
    raw_summary_path = OUTPUT_DIR / 'raw_categorical_summary.json'
    processed_path = OUTPUT_DIR / 'train.parquet'
    resampled_path = OUTPUT_DIR / 'train_adasyn_tomek.parquet'
    preview_path = OUTPUT_DIR / 'adasyn_generated_preview.parquet'
    MAX_UNIQUE_EXAMPLES = 12

    if not all(p.exists() for p in [raw_summary_path, processed_path,
                                    resampled_path, preview_path]):
        print('Run the updated main export, ADASYN and Tomek Links cells first. '
              'They create the categorical baseline and generated-row preview '
              'inside OUTPUT_DIR. This test does not rerun resampling.')
    else:
        category_baseline = json.loads(raw_summary_path.read_text())
        raw_category_uniques = pd.DataFrame(category_baseline['raw_features'])
        print('Raw categorical values from the SAME training split before processing:')
        display(raw_category_uniques)
        print('Rows compared: raw/processed training = '
              f"{category_baseline['training_rows']:,}; final resampled training = "
              f'{pq.ParquetFile(resampled_path).metadata.num_rows:,}')

        # Read one encoded column at a time, not either complete feature matrix.
        comparison_rows = []
        for mapping in category_baseline['encoded_features']:
            column = mapping['Encoded column']
            before_unique = pc.unique(pq.read_table(
                processed_path, columns=[column]
            ).column(0).cast(pa.float64()))
            after_unique = pc.unique(pq.read_table(
                resampled_path, columns=[column]
            ).column(0).cast(pa.float64()))
            new_unique = pc.filter(
                after_unique,
                pc.invert(pc.is_in(after_unique, value_set=before_unique)),
            )
            non_binary = None
            if mapping['Encoding'] != 'Joint identifier PCA':
                non_binary = int(pc.sum(pc.and_(
                    pc.not_equal(after_unique, 0),
                    pc.not_equal(after_unique, 1),
                )).as_py() or 0)
            comparison_rows.append({
                **mapping,
                'Unique after processing': len(before_unique),
                'Processed values (preview)': before_unique.slice(0, MAX_UNIQUE_EXAMPLES).to_pylist(),
                'Unique after ADASYN + Tomek': len(after_unique),
                'Resampled values (preview)': after_unique.slice(0, MAX_UNIQUE_EXAMPLES).to_pylist(),
                'New unique entries': len(new_unique),
                'New values (preview)': new_unique.slice(0, MAX_UNIQUE_EXAMPLES).to_pylist(),
                'Non-binary unique entries (final)': non_binary,
            })
            del before_unique, after_unique, new_unique

        category_unique_comparison = pd.DataFrame(comparison_rows)
        print('Before processing -> encoded training -> final resampled training:')
        display(category_unique_comparison)
        print('Indicator columns containing values other than 0 and 1 after resampling:')
        display(category_unique_comparison.loc[
            category_unique_comparison['Non-binary unique entries (final)'].fillna(0).gt(0)
        ].head(30))

        generated_samples = pd.read_parquet(preview_path)
        if generated_samples.empty:
            print('ADASYN generated no rows in this run; the generated preview is empty.')
        else:
            encoded_columns = [m['Encoded column'] for m in category_baseline['encoded_features']]
            indicator_columns = [m['Encoded column'] for m in category_baseline['encoded_features']
                                 if m['Encoding'] != 'Joint identifier PCA']
            generated_encoded_values = generated_samples.melt(
                id_vars=['ADASYN row', 'Attack', 'Label', 'Kept after Tomek'],
                value_vars=indicator_columns,
                var_name='Encoded column', value_name='Generated value',
            )
            generated_fractional_examples = generated_encoded_values.loc[
                ~generated_encoded_values['Generated value'].isin([0, 1])
            ]
            focus_columns = list(dict.fromkeys(
                generated_fractional_examples['Encoded column'].tolist()
                + encoded_columns
            ))[:12]
            print('Genuine ADASYN-generated rows (up to 5 per oversampled Attack):')
            display(generated_samples[
                ['ADASYN row', 'Attack', 'Label', 'Kept after Tomek'] + focus_columns
            ].head(20))
            print('Generated non-binary indicator coordinates (first 40):')
            display(generated_fractional_examples.head(40))
            if generated_fractional_examples.empty:
                print('No fractional indicator values occurred in this small preview. '
                      'Use the full-data unique comparison above for all final rows.')


Raw categorical values from the SAME training split before processing:


,Raw feature,Raw unique entries,Raw values (first 20),Raw missing rows
0,PROTOCOL,5,"[6, 17, 1, 2, 58]",0
1,L7_PROTO,136,"[7.0, 91.0, 0.0, 131.0, 7.178, 238.0, 13.0, 88.0, 29.0, 91.212, 1.0, 178.0, 47.0, 41.0, 5.0, 79.0, 42.0, 114.0, 112.0, 89.7]",0
2,ICMP_IPV4_TYPE,256,"[0, 49, 131, 46, 54, 5, 126, 145, 22, 4, 47, 113, 51, 136, 26, 58, 160, 171, 178, 3]",0
3,DNS_QUERY_TYPE,11,"[0, 1, 28, 255, 12, 2, 43, 33, 6, 48, 16]",0
4,FTP_COMMAND_RET_CODE,11,"[0.0, 331.0, 530.0, 250.0, 230.0, 229.0, 550.0, 227.0, 221.0, 200.0, 150.0]",0
5,L4_SRC_PORT,52252,"[40470, 443, 47142, 57259, 48932, 34612, 43456, 15207, 38266, 2984, 36017, 56504, 54330, 40878, 39477, 38027, 42538, 49328, 42549, 36414]",0
6,L4_DST_PORT,38635,"[80, 40308, 53, 1074, 1782, 53823, 20826, 2984, 2099, 443, 6668, 1023, 1043, 5679, 1556, 8080, 7911, 51493, 8022, 32774]",0
7,IPV4_SRC_ADDR,310,"[192.168.1.36, 192.168.1.184, 192.168.1.30, 192.168.1.31, 192.168.1.79, 192.168.1.32, 192.168.1.39, 192.168.1.38, 192.168.1.34, 192.168.1.193, 192.168.1.35, 192.168.1.33, 192.168.1.195, 192.168.1.190, 192.168.1.194, 192.168.1.169, 192.168.1.37, 192.168.1.1, 192.168.1.152, 192.168.1.180]",0
8,IPV4_DST_ADDR,1018,"[192.168.1.190, 192.168.1.31, 192.168.1.1, 192.168.1.180, 192.168.1.49, 192.168.1.195, 192.168.1.184, 192.168.1.152, 192.168.1.194, 18.184.104.180, 192.168.1.37, 18.194.169.124, 192.168.1.186, 52.28.231.150, 192.168.1.46, 176.28.50.165, 192.168.35.192, 192.168.1.79, 224.0.0.252, 192.168.35.168]",0
9,DNS_QUERY_ID,51974,"[0, 49467, 33158, 29273, 60701, 47307, 30844, 23865, 26083, 8134, 39572, 55875, 55055, 17282, 58174, 17133, 49608, 111, 1829, 14834]",0


Rows compared: raw/processed training = 509,569; final resampled training = 594,442
Before processing -> encoded training -> final resampled training:


,Raw feature(s),Raw unique entries,Raw values (preview),Encoding,Encoded column,Unique after processing,Processed values (preview),Unique after ADASYN + Tomek,Resampled values (preview),New unique entries,New values (preview),Non-binary unique entries (final)
0,PROTOCOL,{'PROTOCOL': 5},"{'PROTOCOL': ['6', '17', '1', '2', '58']}",One-hot,categories__onehot__PROTOCOL_1,2,"[0.0, 1.0]",2,"[0.0, 1.0]",0,[],0.0
1,PROTOCOL,{'PROTOCOL': 5},"{'PROTOCOL': ['6', '17', '1', '2', '58']}",One-hot,categories__onehot__PROTOCOL_2,2,"[0.0, 1.0]",2,"[0.0, 1.0]",0,[],0.0
2,PROTOCOL,{'PROTOCOL': 5},"{'PROTOCOL': ['6', '17', '1', '2', '58']}",One-hot,categories__onehot__PROTOCOL_6,2,"[1.0, 0.0]",2549,"[1.0, 0.0, 0.47961547373387403, 0.10014913831247108, 0.6165574768407944, 0.38830614107943817, 0.9838729845448108, 0.06623098846553066, 0.5777863032306767, 0.9895683115440449, 0.739769631642193, 0.03209114457948592]",2547,"[0.47961547373387403, 0.10014913831247108, 0.6165574768407944, 0.38830614107943817, 0.9838729845448108, 0.06623098846553066, 0.5777863032306767, 0.9895683115440449, 0.739769631642193, 0.03209114457948592, 0.0213631630969624, 0.03611011198219283]",2547.0
3,PROTOCOL,{'PROTOCOL': 5},"{'PROTOCOL': ['6', '17', '1', '2', '58']}",One-hot,categories__onehot__PROTOCOL_17,2,"[0.0, 1.0]",2549,"[0.0, 1.0, 0.520384526266126, 0.8998508616875289, 0.3834425231592056, 0.6116938589205618, 0.016127015455189242, 0.9337690115344693, 0.4222136967693233, 0.010431688455955102, 0.260230368357807, 0.9679088554205141]",2547,"[0.520384526266126, 0.8998508616875289, 0.3834425231592056, 0.6116938589205618, 0.016127015455189242, 0.9337690115344693, 0.4222136967693233, 0.010431688455955102, 0.260230368357807, 0.9679088554205141, 0.9786368369030376, 0.9638898880178072]",2547.0
4,PROTOCOL,{'PROTOCOL': 5},"{'PROTOCOL': ['6', '17', '1', '2', '58']}",One-hot,categories__onehot__PROTOCOL_58,2,"[0.0, 1.0]",2,"[0.0, 1.0]",0,[],0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
476,SERVER_TCP_FLAGS,{'SERVER_TCP_FLAGS': 20},"{'SERVER_TCP_FLAGS': ['27', '0', '20', '18', '19', '4', '30', '26', '25', '16', '24', '22', '17', '31', '23', '28', '29', '21', '2', '91']}",TCP flag bits,categories__flags__SERVER_TCP_FLAGS_ACK,2,"[1.0, 0.0]",1040,"[1.0, 0.0, 0.47961547373387403, 0.10014913831247108, 0.6165574768407944, 0.38830614107943817, 0.9838729845448108, 0.06623098846553066, 0.5777863032306767, 0.9895683115440449, 0.739769631642193, 0.03209114457948592]",1038,"[0.47961547373387403, 0.10014913831247108, 0.6165574768407944, 0.38830614107943817, 0.9838729845448108, 0.06623098846553066, 0.5777863032306767, 0.9895683115440449, 0.739769631642193, 0.03209114457948592, 0.0213631630969624, 0.03611011198219283]",1038.0
477,SERVER_TCP_FLAGS,{'SERVER_TCP_FLAGS': 20},"{'SERVER_TCP_FLAGS': ['27', '0', '20', '18', '19', '4', '30', '26', '25', '16', '24', '22', '17', '31', '23', '28', '29', '21', '2', '91']}",TCP flag bits,categories__flags__SERVER_TCP_FLAGS_URG,1,[0.0],1,[0.0],0,[],0.0
478,SERVER_TCP_FLAGS,{'SERVER_TCP_FLAGS': 20},"{'SERVER_TCP_FLAGS': ['27', '0', '20', '18', '19', '4', '30', '26', '25', '16', '24', '22', '17', '31', '23', '28', '29', '21', '2', '91']}",TCP flag bits,categories__flags__SERVER_TCP_FLAGS_ECE,2,"[0.0, 1.0]",2,"[0.0, 1.0]",0,[],0.0
479,SERVER_TCP_FLAGS,{'SERVER_TCP_FLAGS': 20},"{'SERVER_TCP_FLAGS': ['27', '0', '20', '18', '19', '4', '30', '26', '25', '16', '24', '22', '17', '31', '23', '28', '29', '21', '2', '91']}",TCP flag bits,categories__flags__SERVER_TCP_FLAGS_CWR,1,[0.0],1,[0.0],0,[],0.0


Indicator columns containing values other than 0 and 1 after resampling:


,Raw feature(s),Raw unique entries,Raw values (preview),Encoding,Encoded column,Unique after processing,Processed values (preview),Unique after ADASYN + Tomek,Resampled values (preview),New unique entries,New values (preview),Non-binary unique entries (final)
2,PROTOCOL,{'PROTOCOL': 5},"{'PROTOCOL': ['6', '17', '1', '2', '58']}",One-hot,categories__onehot__PROTOCOL_6,2,"[1.0, 0.0]",2549,"[1.0, 0.0, 0.47961547373387403, 0.10014913831247108, 0.6165574768407944, 0.38830614107943817, 0.9838729845448108, 0.06623098846553066, 0.5777863032306767, 0.9895683115440449, 0.739769631642193, 0.03209114457948592]",2547,"[0.47961547373387403, 0.10014913831247108, 0.6165574768407944, 0.38830614107943817, 0.9838729845448108, 0.06623098846553066, 0.5777863032306767, 0.9895683115440449, 0.739769631642193, 0.03209114457948592, 0.0213631630969624, 0.03611011198219283]",2547.0
3,PROTOCOL,{'PROTOCOL': 5},"{'PROTOCOL': ['6', '17', '1', '2', '58']}",One-hot,categories__onehot__PROTOCOL_17,2,"[0.0, 1.0]",2549,"[0.0, 1.0, 0.520384526266126, 0.8998508616875289, 0.3834425231592056, 0.6116938589205618, 0.016127015455189242, 0.9337690115344693, 0.4222136967693233, 0.010431688455955102, 0.260230368357807, 0.9679088554205141]",2547,"[0.520384526266126, 0.8998508616875289, 0.3834425231592056, 0.6116938589205618, 0.016127015455189242, 0.9337690115344693, 0.4222136967693233, 0.010431688455955102, 0.260230368357807, 0.9679088554205141, 0.9786368369030376, 0.9638898880178072]",2547.0
5,L7_PROTO,{'L7_PROTO': 136},"{'L7_PROTO': ['7.0', '91.0', '0.0', '131.0', '7.178', '238.0', '13.0', '88.0', '29.0', '91.212', '1.0', '178.0', '47.0', '41.0', '5.0', '79.0', '42.0', '114.0', '112.0', '89.7']}",One-hot,categories__onehot__L7_PROTO_0.0,2,"[0.0, 1.0]",9934,"[0.0, 1.0, 0.6346804368351566, 0.940597026109229, 0.6277413581052478, 0.8593836431177161, 0.8767537277497094, 0.13699211850079906, 0.022985740400105703, 0.8014713096145859, 0.8896777031209219, 0.42302834522320487]",9932,"[0.6346804368351566, 0.940597026109229, 0.6277413581052478, 0.8593836431177161, 0.8767537277497094, 0.13699211850079906, 0.022985740400105703, 0.8014713096145859, 0.8896777031209219, 0.42302834522320487, 0.9200140420593034, 0.39754673683640507]",9932.0
12,L7_PROTO,{'L7_PROTO': 136},"{'L7_PROTO': ['7.0', '91.0', '0.0', '131.0', '7.178', '238.0', '13.0', '88.0', '29.0', '91.212', '1.0', '178.0', '47.0', '41.0', '5.0', '79.0', '42.0', '114.0', '112.0', '89.7']}",One-hot,categories__onehot__L7_PROTO_5.119,2,"[0.0, 1.0]",153,"[0.0, 1.0, 0.07986080842414955, 0.7452398656047523, 0.4575168625646586, 0.19238134126683104, 0.6789147012854193, 0.7138251342122597, 0.5889347967216513, 0.4494251378321674, 0.5136446241539315, 0.17966140277789933]",151,"[0.07986080842414955, 0.7452398656047523, 0.4575168625646586, 0.19238134126683104, 0.6789147012854193, 0.7138251342122597, 0.5889347967216513, 0.4494251378321674, 0.5136446241539315, 0.17966140277789933, 0.8470811200670954, 0.7588838022782424]",151.0
13,L7_PROTO,{'L7_PROTO': 136},"{'L7_PROTO': ['7.0', '91.0', '0.0', '131.0', '7.178', '238.0', '13.0', '88.0', '29.0', '91.212', '1.0', '178.0', '47.0', '41.0', '5.0', '79.0', '42.0', '114.0', '112.0', '89.7']}",One-hot,categories__onehot__L7_PROTO_5.12,2,"[0.0, 1.0]",105,"[0.0, 1.0, 0.4672365054631783, 0.6317485069007955, 0.1693310410020814, 0.7205615749722859, 0.848143908964426, 0.5926787055286639, 0.012896081982758156, 0.3982186266439607, 0.85192675876999, 0.8281354186597312]",103,"[0.4672365054631783, 0.6317485069007955, 0.1693310410020814, 0.7205615749722859, 0.848143908964426, 0.5926787055286639, 0.012896081982758156, 0.3982186266439607, 0.85192675876999, 0.8281354186597312, 0.28517240552724155, 0.5566672901407151]",103.0
16,L7_PROTO,{'L7_PROTO': 136},"{'L7_PROTO': ['7.0', '91.0', '0.0', '131.0', '7.178', '238.0', '13.0', '88.0', '29.0', '91.212', '1.0', '178.0', '47.0', '41.0', '5.0', '79.0', '42.0', '114.0', '112.0', '89.7']}",One-hot,categories__onehot__L7_PROTO_5.126,2,"[0.0, 1.0]",488,"[0.0, 1.0, 0.902998954599468

Genuine ADASYN-generated rows (up to 5 per oversampled Attack):


,ADASYN row,Attack,Label,Kept after Tomek,categories__onehot__L7_PROTO_0.0,categories__onehot__L7_PROTO_41.0,categories__onehot__DNS_QUERY_TYPE_1,categories__onehot__DNS_QUERY_TYPE_28,categories__binary__L4_SRC_PORT_0,categories__binary__L4_SRC_PORT_1,categories__binary__L4_SRC_PORT_2,categories__binary__L4_SRC_PORT_4,categories__binary__L4_SRC_PORT_5,categories__binary__L4_SRC_PORT_6,categories__binary__L4_SRC_PORT_7,categories__binary__L4_SRC_PORT_8
0,509569,Backdoor,1,True,1.000000,0.000000,1.000000,0.000000,0.325147,0.674853,0.000000,0.325147,0.000000,1.000000,0.674853,1.000000
1,509570,Backdoor,1,True,1.000000,0.000000,0.367285,0.632715,0.000000,1.000000,0.000000,0.000000,0.632715,1.000000,1.000000,1.000000
2,509571,Backdoor,1,True,1.000000,0.000000,0.165993,0.834007,0.000000,0.165993,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
3,509572,Backdoor,1,True,1.000000,0.000000,0.828889,0.171111,0.000000,1.000000,0.000000,0.000000,0.171111,1.000000,1.000000,1.000000
4,509573,Backdoor,1,True,1.000000,0.000000,0.069127,0.930873,0.000000,1.000000,0.000000,0.000000,0.930873,1.000000,1.000000,1.000000
5,532801,mitm,1,True,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,532802,mitm,1,True,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,532803,mitm,1,True,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,532804,mitm,1,True,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,532805,mitm,1,True,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


Generated non-binary indicator coordinates (first 40):


,ADASYN row,Attack,Label,Kept after Tomek,Encoded column,Generated value
89,562592,ransomware,1,True,categories__onehot__L7_PROTO_0.0,0.448707
824,562592,ransomware,1,True,categories__onehot__L7_PROTO_41.0,0.551293
5971,509570,Backdoor,1,True,categories__onehot__DNS_QUERY_TYPE_1,0.367285
5972,509571,Backdoor,1,True,categories__onehot__DNS_QUERY_TYPE_1,0.165993
5973,509572,Backdoor,1,True,categories__onehot__DNS_QUERY_TYPE_1,0.828889
5974,509573,Backdoor,1,True,categories__onehot__DNS_QUERY_TYPE_1,0.069127
6046,509570,Backdoor,1,True,categories__onehot__DNS_QUERY_TYPE_28,0.632715
6047,509571,Backdoor,1,True,categories__onehot__DNS_QUERY_TYPE_28,0.834007
6048,509572,Backdoor,1,True,categories__onehot__DNS_QUERY_TYPE_28,0.171111
6049,509573,Backdoor,1,True,categories__onehot__DNS_QUERY_TYPE_28,0.930873


### Baseline model and three partial-balancing policies

Enable RUN_THRESHOLD_TESTS to compare an unresampled baseline with selection=target ratios 0.25, 0.5 and 0.75. These are paired policy comparisons, not a selection-only ablation. All cases use the same FULL prepared training input and untouched prepared validation set. Attack is the multiclass target; neither Attack nor Label enters the features. Test/zero-day data is not read.

Each fresh model uses one 32-unit ReLU layer, Adam, learning rate 0.001, alpha 1e-4, seed SEED and 30 epochs. The epoch budget is equal; larger training sets have more optimizer updates per epoch. Reports include accuracy, balanced accuracy, macro/weighted F1, per-Attack precision/recall/F1, validation support and confusion matrices. Macro F1 uses the fixed training-class list; inspect support for classes absent from validation.

Counts, generated rows, post-ADASYN matrix GiB (not process peak RAM), removed rows, sampler/training times and compute total are saved. Checkpoint/result reuse is explicit; cached durations describe original compute. Each policy owns a compatible checkpoint, including reuse of the main 0.5 ADASYN output. Successful models/reports are reused only when source train/validation, policy and model settings match. Errors are saved without substituting another sampler. These tests are optional and can be expensive.

Outputs in threshold_tests include comparison.csv, per-case model.joblib, metrics.json, class counts, generated previews and CSV/HTML confusion matrices. HTML plots use Plotly CDN. Choose thresholds on validation outcomes; reserve test data for a final evaluation.

Elapsed times in tables, CSVs, metrics JSON and prints use hours/minutes/seconds. Numeric seconds stay inside checkpoints for arithmetic; compatible older result files with second-based metrics are converted when reused.


In [57]:
if RUN_THRESHOLD_TESTS:
    import gc
    import json
    import joblib
    import numpy as np
    import pandas as pd
    import plotly.express as px
    import pyarrow.parquet as pq
    from time import perf_counter
    from sklearn.neural_network import MLPClassifier
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                                 classification_report, confusion_matrix)
    from imblearn.over_sampling import ADASYN
    from imblearn.under_sampling import TomekLinks

    THRESHOLDS = [0.25, 0.50, 0.75]  # Selection AND target move together.
    THRESHOLD_NN_EPOCHS = 30
    REUSE_THRESHOLD_RESULTS = True
    test_dir = OUTPUT_DIR / 'threshold_tests'
    test_dir.mkdir(parents=True, exist_ok=True)
    X_base, y_base, test_features, train_metadata = read_prepared_matrix(OUTPUT_DIR / 'train.parquet')
    X_validation, y_validation, _, _ = read_prepared_matrix(
        OUTPUT_DIR / 'val.parquet', test_features,
    )
    test_classes = np.unique(y_base)
    test_before = pd.Series(y_base).value_counts().sort_index()
    test_majority = test_before.idxmax()
    test_target_dtypes = train_metadata['target_dtypes']
    test_label_map = train_metadata['attack_to_label']
    del train_metadata
    # Recheck training content if the prepared file changed since main resampling.
    test_policy_settings = {**current_policy_settings,
                            'train_sha256': prepared_signature(OUTPUT_DIR / 'train.parquet'),
                            'features': test_features}
    validation_hash = prepared_signature(OUTPUT_DIR / 'val.parquet')
    threshold_rows = []

    for ratio in [None] + THRESHOLDS:
        case = 'baseline' if ratio is None else f'ratio_{ratio:g}'
        case_dir = test_dir / case
        case_dir.mkdir(parents=True, exist_ok=True)
        settings = {**test_policy_settings, 'selection': ratio or SELECT_BELOW_RATIO,
                    'target': ratio or TARGET_RATIO}
        model_settings = {
            'policy': settings, 'baseline': ratio is None, 'validation_sha256': validation_hash,
            'hidden': [32], 'epochs': THRESHOLD_NN_EPOCHS, 'seed': SEED,
            'learning_rate': 0.001, 'alpha': 1e-4,
        }
        metrics_path = case_dir / 'metrics.json'
        case_started = perf_counter()
        if REUSE_THRESHOLD_RESULTS and metrics_path.exists():
            previous = json.loads(metrics_path.read_text())
            required = ['model.joblib', 'per_attack_report.csv', 'class_counts.csv',
                        'confusion_matrix.csv', 'confusion_matrix.html']
            if previous['settings'] == model_settings and all((case_dir / name).exists() for name in required):
                row = previous['metrics'].copy()
                # Older saved results use numeric second columns; display them as h/m/s too.
                for name in list(row):
                    if name.endswith(' seconds'):
                        row[name.removesuffix(' seconds') + ' time'] = format_duration(row.pop(name))
                row['Result reused'] = True
                row['Original run wall time'] = row.get('Original run wall time', row['This run wall time'])
                row['This run wall time'] = format_duration(perf_counter() - case_started)
                metrics_path.write_text(json.dumps({'settings': model_settings, 'metrics': row}, indent=2))
                threshold_rows.append(row)
                print('Reusing completed test:', case, '(stored compute times)', flush=True)
                continue
        wall_started = perf_counter()
        state = None
        try:
            print(f'Threshold test: {case}', flush=True)
            adasyn_time = tomek_time = 0.0
            adasyn_cached = False
            adasyn_backend_used = 'not used'
            requested_counts = test_before.to_dict()
            if ratio is None:
                X_fit, y_fit = X_base, y_base
                after_adasyn = test_before.copy()
                adasyn_matrix_gib = X_base.nbytes / 2**30
                added = removed = 0
            else:
                requested = math.ceil(ratio * int(test_before.max()))
                targets = {attack: requested for attack, count in test_before.items()
                           if count < ratio * test_before.max()
                           and count < (1 - ADASYN_BALANCE_TOLERANCE) * requested}
                requested_counts.update(targets)
                policy_dir = OUTPUT_DIR / 'resampling' / f'select_{ratio:g}_target_{ratio:g}'
                checkpoint = policy_dir / 'adasyn_checkpoint.joblib'
                if REUSE_ADASYN_CHECKPOINT and checkpoint.exists():
                    payload = joblib.load(checkpoint, mmap_mode='r')
                    if payload['settings'] != settings:
                        raise ValueError('Checkpoint settings/data differ. Set REUSE_ADASYN_CHECKPOINT=False to recompute.')
                    state = payload['state']
                    del payload
                    adasyn_cached = True
                else:
                    started = perf_counter()
                    if targets:
                        smallest = min(int(test_before[a]) for a in targets)
                        if smallest < 2:
                            raise ValueError('ADASYN needs at least two training examples per targeted Attack.')
                        print(f'{case} ADASYN: starting...', flush=True)
                        X_after, y_after = ADASYN(
                            sampling_strategy=targets, n_neighbors=min(5, smallest - 1),
                            random_state=SEED,
                        ).fit_resample(X_base, y_base)
                        adasyn_time = perf_counter() - started
                    else:
                        X_after, y_after = X_base, y_base
                    state = {
                        'X_adasyn': X_after, 'y_adasyn': y_after,
                        'resampling_features': test_features,
                        'original_columns': pq.read_schema(OUTPUT_DIR / 'train.parquet').names,
                        'target_dtypes': test_target_dtypes, 'attack_to_label': test_label_map,
                        'before_counts': test_before, 'adasyn_counts': pd.Series(y_after).value_counts().sort_index(),
                        'majority_attack': test_majority, 'input_train_rows': len(y_base),
                        'adasyn_rows': len(y_after), 'adasyn_seconds': adasyn_time,
                        'adasyn_skipped': not bool(targets), 'sampling_targets': targets,
                        'adasyn_backend': GPU_STATUS if targets else 'skipped',
                        'generated_preview': generated_row_preview(
                            X_after, y_after, len(y_base), targets, test_features, test_label_map,
                        ),
                    }
                    atomic_checkpoint(checkpoint, state, settings)
                    del X_after, y_after
                X_after, y_after = state['X_adasyn'], state['y_adasyn']
                adasyn_time = state['adasyn_seconds']
                adasyn_backend_used = state['adasyn_backend']
                adasyn_matrix_gib = X_after.nbytes / 2**30
                added = len(y_after) - len(y_base)
                after_adasyn = pd.Series(y_after).value_counts().sort_index()
                started = perf_counter()
                print(f'{case} Tomek Links: starting...', flush=True)
                case_tomek = TomekLinks(sampling_strategy=[test_majority], n_jobs=-1)
                X_fit, y_fit = case_tomek.fit_resample(X_after, y_after)
                tomek_time = perf_counter() - started
                removed = len(y_after) - len(y_fit)
                preview = state['generated_preview'].copy()
                preview['Kept after Tomek'] = np.isin(preview['ADASYN row'], case_tomek.sample_indices_)
                preview_path = case_dir / 'generated_preview.parquet'
                preview.to_parquet(preview_path, index=False)
                del state, X_after, y_after, preview
                state = None
            counts = pd.DataFrame({
                'Before': test_before, 'Requested': pd.Series(requested_counts),
                'After ADASYN': after_adasyn, 'After Tomek': pd.Series(y_fit).value_counts(),
            }).fillna(0).astype(int).rename_axis('Attack')
            counts_path = case_dir / 'class_counts.csv'
            counts.to_csv(counts_path)
            print(f'{case}: training identical 32-unit MLP for {THRESHOLD_NN_EPOCHS} epochs; '
                  f'X={X_fit.shape}', flush=True)
            model = MLPClassifier(
                hidden_layer_sizes=(32,), activation='relu', solver='adam',
                batch_size=min(1024, len(y_fit)), learning_rate_init=0.001,
                alpha=1e-4, random_state=SEED,
            )
            started = perf_counter()
            for epoch in range(THRESHOLD_NN_EPOCHS):
                model.partial_fit(X_fit, y_fit, classes=test_classes)
            train_time = perf_counter() - started
            prediction = model.predict(X_validation)
            row = {
                'Case': case, 'Status': 'finished', 'Selection': ratio, 'Target': ratio,
                'Training rows': len(y_fit), 'Generated rows': added, 'Tomek removed': removed,
                'Post-ADASYN GiB': adasyn_matrix_gib, 'ADASYN time': format_duration(adasyn_time),
                'Tomek time': format_duration(tomek_time), 'Training time': format_duration(train_time),
                'Compute time': format_duration(adasyn_time + tomek_time + train_time),
                'Accuracy': float(accuracy_score(y_validation, prediction)),
                'Balanced accuracy': float(balanced_accuracy_score(y_validation, prediction)),
                'Macro F1': float(f1_score(y_validation, prediction, labels=test_classes, average='macro', zero_division=0)),
                'Weighted F1': float(f1_score(y_validation, prediction, labels=test_classes, average='weighted', zero_division=0)),
                'ADASYN cached': adasyn_cached, 'ADASYN backend': adasyn_backend_used,
                'Tomek backend': GPU_STATUS if ratio is not None else 'not used',
                'Result reused': False, 'This run wall time': format_duration(perf_counter() - wall_started),
            }
            report = pd.DataFrame(classification_report(
                y_validation, prediction, labels=test_classes, output_dict=True, zero_division=0,
            )).T
            report_path = case_dir / 'per_attack_report.csv'
            report.to_csv(report_path)
            matrix = confusion_matrix(y_validation, prediction, labels=test_classes)
            matrix_path = case_dir / 'confusion_matrix.csv'
            pd.DataFrame(matrix, index=test_classes, columns=test_classes).to_csv(matrix_path)
            fig = px.imshow(matrix, x=test_classes, y=test_classes, text_auto=True,
                            labels={'x': 'Predicted Attack', 'y': 'True Attack'}, title=case)
            figure_path = case_dir / 'confusion_matrix.html'
            fig.write_html(figure_path, include_plotlyjs='cdn')
            fig.show()
            model_path = case_dir / 'model.joblib'
            joblib.dump(model, model_path)
            metrics_path.write_text(json.dumps({'settings': model_settings, 'metrics': row}, indent=2))
            threshold_rows.append(row)
        except (ValueError, RuntimeError, MemoryError) as error:
            # A failed sampler is reported, never silently replaced by another method.
            row = {'Case': case, 'Status': 'failed', 'Error': str(error), 'Result reused': False}
            threshold_rows.append(row)
            error_path = case_dir / 'error.json'
            error_path.write_text(json.dumps(row, indent=2))
            print(case, 'failed:', error, flush=True)
        finally:
            state = None
            for variable in ['X_fit', 'y_fit', 'X_after', 'y_after', 'model', 'case_tomek']:
                globals().pop(variable, None)
            gc.collect()
        # Keep successful partial results if a later configuration fails.
        threshold_results = pd.DataFrame(threshold_rows)
        summary_path = test_dir / 'comparison.csv'
        threshold_results.to_csv(summary_path, index=False)

    threshold_results = pd.DataFrame(threshold_rows)
    summary_path = test_dir / 'comparison.csv'
    threshold_results.to_csv(summary_path, index=False)
    display(threshold_results)
    class_count_frames = []
    for case in threshold_results['Case']:
        path = test_dir / case / 'class_counts.csv'
        if path.exists():
            frame = pd.read_csv(path)
            frame['Case'] = case
            class_count_frames.append(frame)
    if class_count_frames:
        count_comparison = pd.concat(class_count_frames, ignore_index=True)
        fig = px.bar(count_comparison, x='Attack', y='After Tomek', color='Case',
                     barmode='group', title='Training class counts: baseline and threshold policies')
        path = test_dir / 'class_distribution.html'
        fig.write_html(path, include_plotlyjs='cdn')
        fig.show()
    del X_base, y_base, X_validation, y_validation
    gc.collect()


Loading 509,569 rows × 510 features; dtype=float64; working arrays ≈ 1.94 GiB; available RAM=29.20 GiB
Loading 109,194 rows × 510 features; dtype=float64; working arrays ≈ 0.42 GiB; available RAM=27.17 GiB
Threshold test: baseline
baseline: training identical 32-unit MLP for 30 epochs; X=(509569, 510)


Threshold test: ratio_0.25
ratio_0.25 ADASYN: starting...
Checkpoint saved: /kaggle/working/project/outputs/NF_TON_IOT/preprocessing/20261116_112333_all_attacks_log1p_IterativeImputer/resampling/select_0.25_target_0.25/adasyn_checkpoint.joblib
ratio_0.25 Tomek Links: starting...
ratio_0.25: training identical 32-unit MLP for 30 epochs; X=(541566, 510)


Threshold test: ratio_0.5
ratio_0.5 Tomek Links: starting...
ratio_0.5: training identical 32-unit MLP for 30 epochs; X=(594442, 510)


Threshold test: ratio_0.75
ratio_0.75 ADASYN: starting...
Checkpoint saved: /kaggle/working/project/outputs/NF_TON_IOT/preprocessing/20261116_112333_all_attacks_log1p_IterativeImputer/resampling/select_0.75_target_0.75/adasyn_checkpoint.joblib
ratio_0.75 Tomek Links: starting...
ratio_0.75: training identical 32-unit MLP for 30 epochs; X=(646562, 510)


,Case,Status,Selection,Target,Training rows,Generated rows,Tomek removed,Post-ADASYN GiB,ADASYN time,Tomek time,...,Compute time,Accuracy,Balanced accuracy,Macro F1,Weighted F1,ADASYN cached,ADASYN backend,Tomek backend,Result reused,This run wall time
0,baseline,finished,NaN,NaN,509569,0,0,1.936258,00h 00m 00.0s,00h 00m 00.0s,...,00h 02m 24.6s,0.920344,0.875713,0.888659,0.918830,False,not used,not used,False,00h 02m 25.8s
1,ratio_0.25,finished,0.25,0.25,541566,32763,766,2.060751,00h 03m 06.4s,01h 26m 16.8s,...,01h 32m 01.3s,0.918622,0.882902,0.886534,0.917914,False,CPU (NVIDIA GPU/CuPy unavailable),CPU (NVIDIA GPU/CuPy unavailable),False,01h 32m 06.5s
2,ratio_0.5,finished,0.50,0.50,594442,85637,764,2.261661,00h 03m 17.5s,01h 45m 24.7s,...,01h 51m 38.5s,0.916754,0.887086,0.882323,0.917514,True,CPU (NVIDIA GPU/CuPy unavailable),CPU (NVIDIA GPU/CuPy unavailable),False,01h 48m 23.0s
3,ratio_0.75,finished,0.75,0.75,646562,137752,759,2.459688,00h 03m 22.9s,02h 03m 58.5s,...,02h 10m 35.2s,0.899839,0.889259,0.864278,0.906352,False,CPU (NVIDIA GPU/CuPy unavailable),CPU (NVIDIA GPU/CuPy unavailable),False,02h 10m 41.6s


In [58]:
if RUN_TESTING or RUN_COMPARISONS:
    from pathlib import Path
    import gc
    import numpy as np
    import pandas as pd
    from sklearn.model_selection import train_test_split

    if not all(name in globals() for name in ('DATA_DIR', 'FILES', 'DATASET', 'SEED', 'HOLD_OUT_ATTACK', 'HELD_OUT_ATTACK')):
        raise RuntimeError('Run the platform/path/dataset configuration cells at the top first. The main preprocessing, fitting and saving cells are not required.')

    # Release previous experiment frames before reloading the full source.
    for old_name in ('testing_df', 'testing_train', 'exp_sample', 'exp_train', 'exp_val',
                     'exp_num_train', 'exp_num_val', 'exp_cat_train', 'exp_cat_val',
                     'X_case_train', 'X_case_val'):
        globals().pop(old_name, None)
    gc.collect()
    testing_path = Path(DATA_DIR) / FILES[DATASET]
    print('Reloading raw dataset for Testing:', testing_path)
    testing_df = pd.read_parquet(testing_path)
    testing_df.columns = testing_df.columns.str.strip()
    if not testing_df.columns.is_unique:
        raise ValueError('Column names must remain unique after stripping whitespace.')
    if not {'Label', 'Attack'}.issubset(testing_df.columns):
        raise ValueError('Testing requires Label and Attack columns.')
    testing_df['Attack'] = testing_df['Attack'].astype('string[pyarrow]').str.strip()
    if testing_df[['Label', 'Attack']].isna().any().any():
        raise ValueError('Missing targets: clean Label/Attack before splitting; targets are never imputed.')
    if not testing_df['Label'].isin([0, 1]).all():
        raise ValueError('The binary comparisons require Label values 0 and 1.')
    print('Raw dataset:', DATASET, testing_df.shape)
    display(testing_df['Attack'].value_counts())

    # Legacy comparison baseline: identifiers are dropped and raw flags/ICMP stay numeric.
    # This does not reproduce the main Encoding branch or overwrite its saved outputs.
    testing_df = testing_df.drop(columns=[
        'Dataset', 'IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'SRC_IP', 'DST_IP', 'DNS_QUERY_ID'
    ], errors='ignore')
    testing_df = testing_df.loc[:, ~testing_df.columns.str.lower().str.startswith(('unnamed:', 'index'))]
    testing_feature_cols = [c for c in testing_df if c not in ('Label', 'Attack')]
    testing_categorical_cols = [c for c in ('PROTOCOL', 'L7_PROTO', 'DNS_QUERY_TYPE', 'FTP_COMMAND_RET_CODE')
                                if c in testing_feature_cols]
    for col in testing_categorical_cols:
        testing_df[col] = testing_df[col].astype('string').fillna('missing').astype('string[pyarrow]').str.strip()
    testing_numeric_cols = [c for c in testing_feature_cols if c not in testing_categorical_cols]
    testing_df[testing_numeric_cols] = testing_df[testing_numeric_cols].replace([np.inf, -np.inf], np.nan)
    testing_df = compress_extreme_values(testing_df)
    if HOLD_OUT_ATTACK:
        testing_df = testing_df.loc[testing_df['Attack'].ne(HELD_OUT_ATTACK)]
    if testing_df['Attack'].value_counts().min() < 2:
        raise ValueError('An eligible Attack class has fewer than two rows; stratification is not possible.')
    testing_train, testing_remainder = train_test_split(
        testing_df, test_size=0.30, stratify=testing_df['Attack'], random_state=SEED
    )
    # Do not load/use the original validation, test or held-out partition in these comparisons.
    del testing_df, testing_remainder
    gc.collect()
    print(f'Testing training pool: {len(testing_train):,} rows; original 30% reserved for validation/test.')
    print('Testing numeric/categorical features:', len(testing_numeric_cols), len(testing_categorical_cols))

In [59]:
if RUN_TESTING:
    import numpy as np
    import pandas as pd
    from sklearn.experimental import enable_iterative_imputer
    from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
    from sklearn.metrics import mean_absolute_error
    from sklearn.preprocessing import StandardScaler

    print("Starting imputation benchmark on a 50k sample...")

    # 1. Take a 50k sample of the numerical features from the training set
    sample_df = testing_train[testing_numeric_cols].sample(n=min(50000, len(testing_train)), random_state=SEED).copy()

    # Drop any rows that already contain NaNs in the original dataset so we have a clean ground truth
    ground_truth_df = sample_df.dropna()
    print(f"Rows with complete data for ground truth: {len(ground_truth_df)}")

    # Scale the ground truth data first so that features with massive ranges don't dominate the error calculation
    scaler = StandardScaler()
    ground_truth_scaled = pd.DataFrame(scaler.fit_transform(ground_truth_df), columns=testing_numeric_cols)

    # 2. Artificially introduce 10% missing values (NaNs) into the scaled ground truth
    np.random.seed(SEED)
    missing_mask = np.random.rand(*ground_truth_scaled.shape) < 0.10
    masked_data = ground_truth_scaled.copy()
    masked_data[missing_mask] = np.nan

    # We only want to calculate the error on the exact cells we masked
    original_values = ground_truth_scaled.values[missing_mask]

    # 3. Define the Imputers
    imputers = {
        'SimpleImputer (Median)': SimpleImputer(strategy='median'),
        'KNNImputer (k=5, weighted)': KNNImputer(n_neighbors=5, weights='distance'),
        'IterativeImputer (MICE)': IterativeImputer(max_iter=10, random_state=SEED)
    }

    # 4. Evaluate each Imputer
    results = {}
    for name, imputer in imputers.items():
        print(f"Evaluating {name}...")
        try:
            imputed_data = imputer.fit_transform(masked_data)
            imputed_values = imputed_data[missing_mask]
            mae = mean_absolute_error(original_values, imputed_values)
            results[name] = mae
            print(f"{name} MAE: {mae:.5f}")
        except Exception as e:
            results[name] = f"Failed: {str(e)}"
            print(f"{name} failed with error: {e}")

    # 5. Summary of Results
    print("\n--- Benchmark Results (Lower MAE is better) ---")
    for name, score in results.items():
        if isinstance(score, float):
            print(f"* {name}: {score:.5f}")
        else:
            print(f"* {name}: {score}")

In [60]:
if RUN_TESTING:
    from sklearn.impute import KNNImputer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import make_pipeline

    print("Setting up the hybrid KNNImputer pipeline...")

    # 1. Initialize the KNNImputer (suitable for smaller fit samples)
    knn_imputer = KNNImputer(n_neighbors=5, weights='distance')

    # 2. Build the ColumnTransformer
    hybrid_preprocessor = ColumnTransformer([
        ('numbers', make_pipeline(knn_imputer, StandardScaler()), testing_numeric_cols),
        ('categories', OneHotEncoder(handle_unknown='ignore'), testing_categorical_cols),
    ])

    # 3. Fit on raw training features; X_train above is already transformed.
    print("Extracting 50k training sample for fitting...")
    X_train_sample = testing_train[testing_feature_cols].sample(n=min(50000, len(testing_train)), random_state=SEED)

    # 4. Fit the preprocessor ONLY on the sample
    print("Fitting preprocessor on the 50k sample...")
    hybrid_preprocessor.fit(X_train_sample)

    # 5. Preview a single batch instead of transforming all four datasets.
    X_preview = hybrid_preprocessor.transform(testing_train[testing_feature_cols].head(1000))
    print('KNN preview:', X_preview.shape)

## Three sequential comparisons — same simple neural network

Enable `RUN_COMPARISONS` at the start of Testing and run all Testing cells in order. The loading cell recreates the original training pool from the same raw source used at the top. A fixed sample of up to **1,000,000 training rows** is split 80/20 once for experiments. All cases use the same experiment training/validation rows. Original validation/test/held-out data are not used.

1. **Encoding:** PROTOCOL and L7_PROTO both one-hot versus both binary. StandardScaler and the original class distribution are fixed; other categories remain one-hot.
2. **Scaling:** selected encoding; StandardScaler versus quantile-to-normal, without resampling.
3. **Imbalance:** selected encoding/scaler; no resampling, random under/oversampling at ratios 0.25/0.5/1.0, **ADASYN**, and **ADASYN followed by Tomek Links**. Targets already reached by the natural ratio are skipped. Ratios use binary Label, not individual attack families.

Every case uses a fresh network with one 32-neuron hidden layer, Adam, **30 epochs**, the same seed and threshold 0.5. Imputation and all preprocessing are fitted on experiment training only, before resampling. Experiment validation is never resampled. Selection uses validation macro-F1, then minority average precision. The existing sample size and network budget are retained; ADASYN/Tomek nearest-neighbour searches may take substantial time at this size. Adjust `SAMPLE_ROWS` explicitly if needed.

**Paper method:** [Abdelkhalek and Mashaly, Section 3.2.3 / Algorithm 2](https://doi.org/10.1007/s11227-023-05073-x): ADASYN generates minority examples; Tomek Links then removes majority-class endpoints. `ADASYN_RATIO=1.0` requests approximate balance and `ADASYN_NEIGHBORS=5` is reduced only if the minority count requires it. The original majority label is kept fixed for cleaning even if ADASYN rounding reverses the class counts. Before/after counts and the removed-row count are displayed.

This is an adaptation to the notebook's **binary Label benchmark**. Benign examples remain present. The paper's separate attack-only multiclass experiment drops benign traffic; that is not applied to binary detection here. ADASYN interpolates in the encoded/scaled feature space, so synthetic categorical coordinates can be fractional and are used only as model inputs. It does not reconstruct raw traffic records. An impossible ADASYN case is reported as skipped; no other sampler is silently substituted. These comparisons do not change the main saved preprocessing outputs.

Implementation references: [ADASYN](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.ADASYN.html), [TomekLinks](https://imbalanced-learn.org/stable/references/generated/imblearn.under_sampling.TomekLinks.html).

The comparisons apply the same fixed log1p compression to the two SECOND_BYTES fields before fitting. They retain their existing legacy feature baseline. They do not evaluate the new main Encoding branch (identifier PCA, binary ports, decomposed flags and ICMP code). Main exports use the Encoding function above.

In [61]:
if RUN_COMPARISONS:
    %pip install -q imbalanced-learn category_encoders
    from time import perf_counter
    import gc
    import numpy as np
    import pandas as pd
    from scipy import sparse
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import OneHotEncoder, StandardScaler, QuantileTransformer
    from sklearn.neural_network import MLPClassifier
    from sklearn.metrics import f1_score, average_precision_score, recall_score, confusion_matrix
    from category_encoders import BinaryEncoder
    from sklearn.experimental import enable_iterative_imputer
    from sklearn.impute import IterativeImputer
    from imblearn.under_sampling import RandomUnderSampler, TomekLinks
    from imblearn.over_sampling import RandomOverSampler, ADASYN

    SAMPLE_ROWS = 1000_000
    NN_EPOCHS = 30  # Same fixed training budget for every fresh neural network.
    RATIOS = [0.25, 0.50, 1.0]
    ADASYN_RATIO = 1.0  # Desired minority/majority ratio; actual counts may differ slightly.
    ADASYN_NEIGHBORS = 5
    # Algorithm 2 removes majority-class endpoints. Freeze the original majority label below.

    # One uniform sample shared by every trial; all eligible source classes retain natural prevalence in expectation.
    exp_sample = testing_train.sample(n=min(SAMPLE_ROWS, len(testing_train)), random_state=SEED).copy()
    exp_train, exp_val = train_test_split(exp_sample, test_size=0.20,
                                         stratify=exp_sample['Label'], random_state=SEED)
    if len(exp_sample['Label'].unique()) != 2:
        raise ValueError('The selected sample must contain both labels. Increase SAMPLE_ROWS.')
    exp_features = list(testing_feature_cols)
    exp_numeric = list(testing_numeric_cols)
    exp_categories = list(testing_categorical_cols)
    exp_protocols = [c for c in exp_features if c.upper() in ('PROTOCOL', 'L7_PROTO')]
    exp_other_categories = [c for c in exp_categories if c not in exp_protocols]
    y_exp_train = exp_train['Label'].to_numpy()
    y_exp_val = exp_val['Label'].to_numpy()
    label_counts = pd.Series(y_exp_train).value_counts()
    if len(label_counts) != 2 or not set(label_counts.index).issubset({0, 1}):
        raise ValueError('These ratio comparisons require binary Label values 0 and 1.')
    minority_label = label_counts.idxmin()
    majority_label = label_counts.idxmax()
    natural_ratio = label_counts.min() / label_counts.max()
    print(f'Sampled {len(exp_sample):,} rows; experiment train {len(exp_train):,}, validation {len(exp_val):,}')
    display(pd.crosstab(exp_sample['Attack'], exp_sample['Label']))
    print('Minority label:', minority_label, '| natural minority/majority ratio:', round(natural_ratio, 4))

In [62]:
if RUN_COMPARISONS:
    # Same imputer and parameters as the main pipeline, fitted anew without experiment-validation leakage.
    exp_imputer = IterativeImputer(max_iter=5, random_state=SEED, skip_complete=True)
    t0 = perf_counter()
    exp_num_train = exp_imputer.fit_transform(exp_train[exp_numeric])
    exp_num_val = exp_imputer.transform(exp_val[exp_numeric])
    print(f'Shared imputation: {format_duration(perf_counter() - t0)}')

    # Keep every other categorical feature unchanged across encoder/scaler experiments.
    if exp_other_categories:
        exp_other_encoder = OneHotEncoder(handle_unknown='ignore', dtype=np.float32)
        exp_cat_train = exp_other_encoder.fit_transform(exp_train[exp_other_categories])
        exp_cat_val = exp_other_encoder.transform(exp_val[exp_other_categories])
    else:
        exp_cat_train = sparse.csr_matrix((len(exp_train), 0), dtype=np.float32)
        exp_cat_val = sparse.csr_matrix((len(exp_val), 0), dtype=np.float32)

    def compare_trial(X_train_trial, X_val_trial, encoding, scaling, sampling='none', ratio=None):
        if ratio is not None and ratio <= natural_ratio:
            print(f'Skip {sampling} ratio {ratio}: natural ratio is already {natural_ratio:.3f}')
            return None
        started = perf_counter()
        adasyn_rows = None
        tomek_removed = 0
        if sampling == 'none':
            X_fit, y_fit = X_train_trial, y_exp_train
        elif sampling in ('under', 'over'):
            sampler_class = RandomUnderSampler if sampling == 'under' else RandomOverSampler
            sampler = sampler_class(sampling_strategy=ratio, random_state=SEED)
            X_fit, y_fit = sampler.fit_resample(X_train_trial, y_exp_train)
        elif sampling in ('adasyn', 'adasyn_tomek'):
            neighbours = min(ADASYN_NEIGHBORS, int(label_counts.min()) - 1)
            if neighbours < 1:
                print(f'Skip {sampling}: ADASYN requires at least two minority examples.')
                return None
            print(f'{sampling}: ADASYN ratio={ratio}, neighbours={neighbours}', flush=True)
            before_counts = pd.Series(y_exp_train).value_counts().sort_index()
            try:
                X_fit, y_fit = ADASYN(
                    sampling_strategy=ratio, n_neighbors=neighbours, random_state=SEED
                ).fit_resample(X_train_trial, y_exp_train)
            except (ValueError, RuntimeError) as error:
                # Separated neighbourhoods or very small requested additions can make ADASYN impossible.
                print(f'Skip {sampling}: {error}')
                return None
            adasyn_rows = len(y_fit)
            count_stages = {'before': before_counts,
                            'after_ADASYN': pd.Series(y_fit).value_counts().sort_index()}
            if sampling == 'adasyn_tomek':
                print('Cleaning ADASYN output with Tomek Links...', flush=True)
                X_fit, y_fit = TomekLinks(
                    sampling_strategy=[majority_label], n_jobs=-1
                ).fit_resample(X_fit, y_fit)
                tomek_removed = adasyn_rows - len(y_fit)
                count_stages['after_Tomek'] = pd.Series(y_fit).value_counts().sort_index()
            display(pd.DataFrame(count_stages).fillna(0).astype(int).rename_axis('Label'))
            print('Tomek rows removed:', tomek_removed)
        else:
            raise ValueError(f'Unknown sampler: {sampling}')
        if len(np.unique(y_fit)) != 2:
            print(f'Skip {sampling}: cleaning left fewer than two classes.')
            return None
        # Fresh, identical small neural network for each case; only input width changes with encoding.
        model = MLPClassifier(hidden_layer_sizes=(32,), activation='relu', solver='adam',
                              batch_size=min(1024, len(y_fit)), learning_rate_init=0.001,
                              alpha=1e-4, random_state=SEED)
        for epoch in range(NN_EPOCHS):
            model.partial_fit(X_fit, y_fit, classes=np.array([0, 1]))
        probability_attack = model.predict_proba(X_val_trial)[:, list(model.classes_).index(1)]
        predicted = (probability_attack >= 0.5).astype(int)
        minority_scores = probability_attack if minority_label == 1 else 1 - probability_attack
        tn, fp, fn, tp = confusion_matrix(y_exp_val, predicted, labels=[0, 1]).ravel()
        fit_counts = pd.Series(y_fit).value_counts()
        result = dict(adasyn_rows=adasyn_rows, tomek_removed=tomek_removed, encoding=encoding, scaling=scaling, sampling=sampling, target_ratio=ratio,
                      actual_ratio=fit_counts.min()/fit_counts.max(), train_rows=len(y_fit),
                      features=X_fit.shape[1], macro_F1=f1_score(y_exp_val, predicted, average='macro'),
                      AP_minority=average_precision_score(y_exp_val == minority_label, minority_scores),
                      recall_minority=recall_score(y_exp_val, predicted, pos_label=minority_label, zero_division=0),
                      recall_attack=recall_score(y_exp_val, predicted, pos_label=1, zero_division=0),
                      false_positive_rate=fp/max(1, fp+tn), fit_time=format_duration(perf_counter()-started),
                      matrix_MB=(X_fit.data.nbytes+X_fit.indices.nbytes+X_fit.indptr.nbytes)/2**20,
                      epochs=NN_EPOCHS)
        print(encoding, scaling, sampling, ratio, '| macro F1:', round(result['macro_F1'], 4))
        return result  # Model and resampled matrix references are released after this call.

    def build_experiment_features(encoding, scaling):
        # Both protocol columns use the SAME encoding in each case; all other categories stay one-hot.
        if len(exp_protocols) != 2:
            raise ValueError('Both PROTOCOL and L7_PROTO are required for this comparison.')
        scaler = (StandardScaler() if scaling == 'standard' else
                  QuantileTransformer(output_distribution='normal', n_quantiles=min(1000, len(exp_train)),
                                      subsample=min(50_000, len(exp_train)), random_state=SEED))
        scaled_train = scaler.fit_transform(exp_num_train)
        scaled_val = scaler.transform(exp_num_val)
        for values in (scaled_train, scaled_val):
            if not np.isfinite(values).all() or np.any(np.abs(values) > np.finfo(np.float32).max):
                raise ValueError('Transformed numeric values cannot be stored safely as float32.')
        num_train = sparse.csr_matrix(scaled_train, dtype=np.float32)
        num_val = sparse.csr_matrix(scaled_val, dtype=np.float32)
        encoder = (OneHotEncoder(handle_unknown='ignore', dtype=np.float32) if encoding == 'one_hot'
                   else BinaryEncoder(cols=exp_protocols, return_df=False,
                                      handle_unknown='value', handle_missing='value'))
        proto_train = sparse.csr_matrix(encoder.fit_transform(exp_train[exp_protocols]), dtype=np.float32)
        proto_val = sparse.csr_matrix(encoder.transform(exp_val[exp_protocols]), dtype=np.float32)
        return (sparse.hstack([num_train, exp_cat_train, proto_train], format='csr'),
                sparse.hstack([num_val, exp_cat_val, proto_val], format='csr'))

    def best_case(results):
        # Same primary metric for all three comparisons; minority AP breaks ties.
        return max(results, key=lambda r: (r['macro_F1'], r['AP_minority']))

### 1. Encoding comparison

In [63]:
if RUN_COMPARISONS:
    # 1. Encoding: BOTH protocol features one-hot versus BOTH binary. No resampling.
    encoding_results = []
    for encoding in ['one_hot', 'binary']:
        X_case_train, X_case_val = build_experiment_features(encoding, 'standard')
        encoding_results.append(compare_trial(X_case_train, X_case_val, encoding, 'standard'))
        del X_case_train, X_case_val
        gc.collect()
    display(pd.DataFrame(encoding_results).round(4))
    BEST_ENCODING = best_case(encoding_results)['encoding']
    print('Selected encoding for the scaler comparison:', BEST_ENCODING)


### 2. Scaler comparison

In [64]:
if RUN_COMPARISONS:
    # 2. Scaling: keep the selected encoding; still no resampling.
    # Reuse its StandardScaler score because the data, model seed and settings are identical.
    scaling_results = [dict(best_case(encoding_results))]
    X_case_train, X_case_val = build_experiment_features(BEST_ENCODING, 'normal')
    scaling_results.append(compare_trial(X_case_train, X_case_val, BEST_ENCODING, 'normal'))
    del X_case_train, X_case_val
    gc.collect()
    display(pd.DataFrame(scaling_results).round(4))
    BEST_SCALER = best_case(scaling_results)['scaling']
    print('Selected encoding/scaler for imbalance testing:', BEST_ENCODING, BEST_SCALER)


### 3. Imbalance comparison — last

In [65]:
if RUN_COMPARISONS:
    # 3. Imbalance LAST: freeze the selected encoding and scaler for every sampling case.
    imbalance_results = [dict(best_case(scaling_results))]  # Unbalanced reference, already evaluated.
    X_case_train, X_case_val = build_experiment_features(BEST_ENCODING, BEST_SCALER)
    for method in ['under', 'over']:
        for ratio in RATIOS:
            result = compare_trial(X_case_train, X_case_val, BEST_ENCODING, BEST_SCALER, method, ratio)
            if result is not None:
                imbalance_results.append(result)
    # Paper comparison: identical experiment rows, preprocessing and network budget.
    for method in ['adasyn', 'adasyn_tomek']:
        result = compare_trial(X_case_train, X_case_val, BEST_ENCODING, BEST_SCALER,
                               method, ADASYN_RATIO)
        if result is not None:
            imbalance_results.append(result)
        gc.collect()
    display(pd.DataFrame(imbalance_results).sort_values(['macro_F1', 'AP_minority'], ascending=False).round(4))
    selected_case = best_case(imbalance_results)
    print('Selected by validation macro-F1:', {k: selected_case[k] for k in
          ['encoding', 'scaling', 'sampling', 'target_ratio', 'macro_F1', 'false_positive_rate']})
    comparison_results = pd.DataFrame(
        [dict(stage='encoding', **r) for r in encoding_results] +
        [dict(stage='scaling', **r) for r in scaling_results] +
        [dict(stage='imbalance', **r) for r in imbalance_results])
    # Free experiment matrices; keep only the small result tables and selected settings.
    del X_case_train, X_case_val, exp_sample, exp_train, exp_val, exp_num_train, exp_num_val
    del exp_cat_train, exp_cat_val, exp_imputer, testing_train
    gc.collect()